# **VIII. Procesamiento de variables categóricas y Evaluación de Nulos**

## Objetivo
Este notebook tiene el propósito de simular y aplicar la agrupación (mapeo) de variables categóricas de alta cardinalidad, así como evaluar y ejecutar la imputación o eliminación de valores nulos. El objetivo crítico es reducir la dimensionalidad del dataset sin alterar la distribución de las variables objetivo (Mortalidad, Severidad, Consumo de Recursos).

## Proceso
1. **Simulación Analítica**: Evaluar el impacto en los *targets* antes de eliminar cualquier fila nula, garantizando que no se elimine información vital de clases minoritarias.
2. **Ingeniería de Diccionarios**: Aplicar diccionarios de homologación para agrupar categorías redundantes o de alta cardinalidad (ej. `PREVISION`, `SERVICIO_SALUD`, `ESPECIALIDAD_MEDICA`).
3. **Binarización Estratégica**: Transformar variables de baja varianza pero alto valor clínico/sociológico (ej. `NACIONALIDAD` a `ES_CHILENO`).
4. **Ejecución Estructural**: Sobrescribir las variables procesadas en los archivos definitivos y eliminar los registros nulos aprobados por la simulación.

In [1]:
import pandas as pd
import os
from IPython.display import display, HTML
import warnings

warnings.filterwarnings('ignore', category=UserWarning)

# ============================================================================
# FUNCIÓN 1: SIMULACIÓN DE MAPEO E IMPACTO
# ============================================================================
def simular_transformacion_categorica(carpeta, archivos, columna, mapeo, valores_nulos):
    """
    Simula el agrupamiento de categorías mediante un diccionario de mapeo y evalúa 
    el impacto de eliminar los valores nulos en los targets.
    OPTIMIZADO: Usa `usecols` para no saturar la memoria RAM.
    """
    print(f"\n[{columna}] Simulando mapeo y evaluando impacto de nulos...\n" + "="*60)

    freq_tot, freq_onco, freq_ctrl = {}, {}, {}
    imp_tot, imp_onco, imp_ctrl = [], [], []

    for archivo in archivos:
        ruta = os.path.join(carpeta, archivo)
        año = archivo[-8:-4]
        
        # OPTIMIZACIÓN DE MEMORIA: Leer solo los encabezados primero
        try:
            encabezados = pd.read_csv(ruta, nrows=0).columns
            if columna not in encabezados:
                print(f"  [!] Columna '{columna}' no encontrada en {año}. Saltando...")
                continue
                
            # Cargar estrictamente lo necesario
            columnas_base = [columna, 'CATEGORIA_CANCER', 'MORTALIDAD', 'SEVERIDAD', 'CONSUMO_RECURSOS']
            columnas_a_leer = [col for col in columnas_base if col in encabezados]
            
            df = pd.read_csv(ruta, usecols=columnas_a_leer, low_memory=False)
            
        except Exception as e:
            print(f"  [!] Error al leer {archivo}: {e}")
            continue

        # 1. Limpieza base y Simulación de Mapeo
        col_clean = df[columna].astype(str).str.strip().str.upper()
        df['MAPPED'] = col_clean.replace(mapeo)
        
        # 2. Etiquetar nulos para visualizarlos juntos
        mask_nulos = df['MAPPED'].isin(valores_nulos)
        df.loc[mask_nulos, 'MAPPED'] = "NULOS_A_ELIMINAR"
        
        # 3. Separación de Cohortes
        mask_control = df['CATEGORIA_CANCER'] == 'SIN_CANCER'
        mask_onco = ~mask_control

        # A. CÁLCULO DE FRECUENCIAS
        freq_tot[año] = df['MAPPED'].value_counts()
        freq_onco[año] = df.loc[mask_onco, 'MAPPED'].value_counts()
        freq_ctrl[año] = df.loc[mask_control, 'MAPPED'].value_counts()

        # B. EVALUACIÓN DE IMPACTO EN TARGETS
        def calcular_impacto(df_sub):
            df_nulos = df_sub[df_sub.index.isin(df[mask_nulos].index)]
            total_nulos = len(df_nulos)
            
            if total_nulos == 0:
                return { "Año": año, "Faltantes": 0 }
            
            mort = df_nulos.get("MORTALIDAD", pd.Series()).value_counts().to_dict()
            sev = df_nulos.get("SEVERIDAD", pd.Series()).value_counts().to_dict()
            cons = df_nulos.get("CONSUMO_RECURSOS", pd.Series()).value_counts().to_dict()
            
            return {
                "Año": año,
                "Faltantes": total_nulos,
                "MORT(0)": mort.get(0, 0), "MORT(1)": mort.get(1, 0),
                "SEV(0)": sev.get(0, 0), "SEV(1)": sev.get(1, 0), "SEV(2)": sev.get(2, 0), "SEV(3)": sev.get(3, 0),
                "CONS(0)": cons.get(0, 0), "CONS(1)": cons.get(1, 0), "CONS(2)": cons.get(2, 0)
            }

        imp_tot.append(calcular_impacto(df))
        imp_onco.append(calcular_impacto(df[mask_onco]))
        imp_ctrl.append(calcular_impacto(df[mask_control]))

    # Consolidar DataFrames
    df_freq = (
        pd.DataFrame(freq_tot).fillna(0).astype(int),
        pd.DataFrame(freq_onco).fillna(0).astype(int),
        pd.DataFrame(freq_ctrl).fillna(0).astype(int)
    )
    
    df_imp = (
        pd.DataFrame(imp_tot).set_index("Año").fillna(0).astype(int),
        pd.DataFrame(imp_onco).set_index("Año").fillna(0).astype(int),
        pd.DataFrame(imp_ctrl).set_index("Año").fillna(0).astype(int)
    )

    return df_freq, df_imp

# ============================================================================
# FUNCIÓN 2: MOSTRAR RESULTADOS
# ============================================================================
def mostrar_resultados(df_freq, df_imp, variable):
    print(f"\nResultados para variable: {variable}\n" + "="*60)
    
    print("\nFrecuencias por Cohorte:")
    print("- Cohorte Total:")
    display(df_freq[0])
    print("- Cohorte Oncológica:")
    display(df_freq[1])
    print("- Cohorte Control:")
    display(df_freq[2])
    
    print("\nImpacto de eliminar valores nulos en targets:")
    print("- Cohorte Total:")
    display(df_imp[0])
    print("- Cohorte Oncológica:")
    display(df_imp[1])
    print("- Cohorte Control:")
    display(df_imp[2])

# ============================================================================
# FUNCIÓN 3: APLICAR TRANSFORMACIÓN DEFINITIVA
# ============================================================================
# ============================================================================
# FUNCIÓN 3: APLICAR TRANSFORMACIÓN DEFINITIVA (ACTUALIZADA)
# ============================================================================
def aplicar_transformacion_categorica(carpeta, archivos, columna, mapeo, valores_nulos, nueva_columna=None):
    """
    Aplica de forma DEFINITIVA un diccionario de mapeo.
    Si se indica 'nueva_columna', conserva la original y guarda los datos mapeados en la nueva.
    Si no se indica, sobrescribe la columna original.
    Elimina los registros que coinciden con los 'valores_nulos' y sobrescribe los archivos CSV.
    """
    col_destino = nueva_columna if nueva_columna else columna
    accion = f"CREANDO NUEVA COLUMNA '{col_destino}'" if nueva_columna else f"SOBRESCRIBIENDO '{columna}'"
    
    print(f"\n[{columna}] EJECUTANDO TRANSFORMACIÓN Y LIMPIEZA DEFINITIVA...")
    print(f"-> {accion}\n" + "="*60)
    
    frecuencias_finales = {}
    total_inicial = 0
    total_final = 0
    
    for archivo in archivos:
        ruta = os.path.join(carpeta, archivo)
        año = archivo[-8:-4]
        
        # Leer el dataset completo
        try:
            df = pd.read_csv(ruta, low_memory=False)
        except ValueError:
            print(f"  [!] Error al leer {archivo}. Saltando...")
            continue
            
        filas_antes = len(df)
        total_inicial += filas_antes
        
        if columna not in df.columns:
            print(f"  [!] Columna {columna} no encontrada en {año}.")
            continue
            
        # Limpieza base de la columna original
        serie_limpia = df[columna].astype(str).str.strip().str.upper()
        
        # Aplicar mapeo y guardar en el destino (nueva o sobrescrita)
        if mapeo:
            df[col_destino] = serie_limpia.replace(mapeo)
        else:
            df[col_destino] = serie_limpia
            
        # Eliminación de nulos basada en la columna DESTINO
        if valores_nulos:
            mask_nulos = df[col_destino].isin(valores_nulos)
            df = df[~mask_nulos]
            
        filas_despues = len(df)
        total_final += filas_despues
        
        # Registrar frecuencias resultantes de la columna DESTINO
        frecuencias_finales[año] = df[col_destino].value_counts()
        
        # Sobrescribir el archivo CSV procesado
        df.to_csv(ruta, index=False, encoding="utf-8-sig")
        
        print(f"  -> {año} | Inicial: {filas_antes:,} | Eliminadas: {filas_antes - filas_despues:,} | Final: {filas_despues:,}")
        
    print("="*60)
    print(f"RESUMEN GLOBAL DE IMPACTO:")
    print(f"  Total filas analizadas: {total_inicial:,}")
    print(f"  Total filas eliminadas: {total_inicial - total_final:,}")
    print(f"  Total filas conservadas: {total_final:,}")
    
    df_freq = pd.DataFrame(frecuencias_finales).fillna(0).astype(int).sort_index()
    
    return df_freq

In [2]:
# 1. Definir rutas y variables
carpeta_procesados = "../../Datos/Datos procesados"
archivos_procesados = [f"GRD_PROCESADO_{año}.csv" for año in range(2019, 2025)]

### **1. Evaluación de Variable Demográfica: Sexo (SEXO)**
- **Objetivo de Simulación**: Evaluar el impacto de descartar registros sin clasificación biológica válida (`DESCONOCIDO`, `NaN`).
- **Análisis de Frecuencias**: La variable se encuentra naturalmente binarizada en las categorías válidas (`HOMBRE`, `MUJER`). No se requiere diccionario de agrupación.
- **Análisis de Impacto (Nulos)**: La simulación revela que la etiqueta `DESCONOCIDO` es estadísticamente marginal. En la cohorte oncológica, la pérdida máxima se registra en 2024 con apenas 19 casos, totalizando 31 episodios descartados en un periodo de 6 años (~0.006% de la muestra). 
- **Veredicto Final**: **Aprobado para eliminación de nulos**. La cantidad de nulos no justifica la creación de reglas de imputación basadas en diagnósticos topográficos sexo-específicos (ej. C51-C58 / C60-C63). Se eliminarán de forma directa las filas con valor `DESCONOCIDO`, preservando la binarización natural de la variable sin comprometer la representación de las clases objetivo (Mortalidad, Severidad, Consumo).

In [3]:
mapeo = {}
nulos = ["DESCONOCIDO"]
frecuencias, impactos = simular_transformacion_categorica(carpeta=carpeta_procesados, archivos=archivos_procesados, columna="SEXO", mapeo=mapeo, valores_nulos=nulos)
mostrar_resultados(frecuencias, impactos, "SEXO")


[SEXO] Simulando mapeo y evaluando impacto de nulos...

Resultados para variable: SEXO

Frecuencias por Cohorte:
- Cohorte Total:


,2019,2020,2021,2022,2023,2024
MAPPED,,,,,,
MUJER,681533,461617,475306,553555,612189,632615
HOMBRE,469646,320137,341540,379207,427329,453067
NULOS_A_ELIMINAR,139,134,63,52,39,116


- Cohorte Oncológica:


,2019,2020,2021,2022,2023,2024
MAPPED,,,,,,
HOMBRE,44510,30448,31882,36100,42120,45910
MUJER,53310,34473,36044,41964,48594,53222
NULOS_A_ELIMINAR,4,4,0,3,1,19


- Cohorte Control:


...

,Faltantes,MORT(0),MORT(1),SEV(0),SEV(1),SEV(2),SEV(3),CONS(0),CONS(1),CONS(2)
Año,,,,,,,,,,
2019,139,130,9,46,61,15,17,62,44,33
2020,134,125,9,13,61,24,36,40,40,54
2021,63,59,4,5,25,16,17,19,26,18
2022,52,48,4,5,26,10,11,18,16,18
2023,39,36,3,5,10,9,15,12,13,14
2024,116,114,2,12,28,57,19,20,42,54


- Cohorte Oncológica:


,Faltantes,MORT(0),MORT(1),SEV(0),SEV(1),SEV(2),SEV(3),CONS(0),CONS(1),CONS(2)
Año,,,,,,,,,,
2019,4,3,1,3,0,0,1,1,0,3
2020,4,4,0,1,2,0,1,0,1,3
2021,0,0,0,0,0,0,0,0,0,0
2022,3,3,0,0,1,2,0,0,0,3
2023,1,1,0,1,0,0,0,1,0,0
2024,19,17,2,1,4,10,4,0,6,13


- Cohorte Control:


,Faltantes,MORT(0),MORT(1),SEV(0),SEV(1),SEV(2),SEV(3),CONS(0),CONS(1),CONS(2)
Año,,,,,,,,,,
2019,135,127,8,43,61,15,16,61,44,30
2020,130,121,9,12,59,24,35,40,39,51
2021,63,59,4,5,25,16,17,19,26,18
2022,49,45,4,5,25,8,11,18,16,15
2023,38,35,3,4,10,9,15,11,13,14
2024,97,97,0,11,24,47,15,20,36,41


In [4]:
aplicar_transformacion_categorica(carpeta=carpeta_procesados, archivos=archivos_procesados, columna="SEXO", mapeo=mapeo, valores_nulos=nulos)


[SEXO] EJECUTANDO TRANSFORMACIÓN Y LIMPIEZA DEFINITIVA...
-> SOBRESCRIBIENDO 'SEXO'
  -> 2019 | Inicial: 1,151,318 | Eliminadas: 139 | Final: 1,151,179
  -> 2020 | Inicial: 781,888 | Eliminadas: 134 | Final: 781,754
  -> 2021 | Inicial: 816,909 | Eliminadas: 63 | Final: 816,846
  -> 2022 | Inicial: 932,814 | Eliminadas: 52 | Final: 932,762
  -> 2023 | Inicial: 1,039,557 | Eliminadas: 39 | Final: 1,039,518
  -> 2024 | Inicial: 1,085,798 | Eliminadas: 116 | Final: 1,085,682
RESUMEN GLOBAL DE IMPACTO:
  Total filas analizadas: 5,808,284
  Total filas eliminadas: 543
  Total filas conservadas: 5,807,741


,2019,2020,2021,2022,2023,2024
SEXO,,,,,,
HOMBRE,469646,320137,341540,379207,427329,453067
MUJER,681533,461617,475306,553555,612189,632615


## Evaluación y Procesamiento de Variable: Etnia (ETNIA)

### 1. Descripción y Justificación
La variable `ETNIA` registra la pertenencia declarada del paciente a pueblos originarios. El análisis inicial reveló tres desafíos críticos que motivaron su transformación estratégica:

* **Ruptura de Codificación Administrativa:** Se detectó un quiebre estructural entre los años 2021 y 2022. Hasta 2021, la categoría mayoritaria era `NINGUNO`; a partir de 2022, dicha categoría desaparece en favor de `OTRO`. La binarización en una clase de control (0) neutraliza este cambio de criterio del registro ministerial.
* **Varianza Cercana a Cero (*Near-Zero Variance*):** Las etnias específicas (Mapuche, Aymara, etc.) representan consistentemente menos del 2% de la muestra anual. Mantener 11 categorías con tan baja representatividad inyectaría ruido excesivo y dispersión (*sparsity*) en los modelos de aprendizaje automático.
* **Integridad de Datos (Encoding):** Se identificaron artefactos de codificación (caracteres mal interpretados como `KAWÃ‰SQAR` o `ATACAMEÃ‘O`) derivados de la exportación original de los archivos, los cuales fueron normalizados en este proceso.

### 2. Estrategia de Transformación (Ingeniería de Características)
Se aplicó una binarización para convertir la alta cardinalidad en una variable dicotómica funcional:

* **Clase 0 (Control / No Pertenencia):** Agrupa los valores `NINGUNO`, `NINGUNA` y `OTRO`. Representa a pacientes que no se identifican con un pueblo originario específico.
* **Clase 1 (Pertenencia a Pueblo Originario):** Agrupa las 9 etnias reconocidas (Mapuche, Aymara, Rapa Nui, Diaguita, Kawésqar, Quechua, Yagán, Colla y Lican Antai). 
* **Gestión de Nulos:** Las etiquetas `DESCONOCIDO` y `NO RESPONDE` se clasificaron como nulos para la evaluación de impacto.

In [5]:
# 1. Mapeo para ETNIA
mapeo = {
    # Clase Control (0)
    "NINGUNO": "0",
    "NINGUNA": "0",
    "OTRO": "0",
    
    # Clase Positiva (1) - Pueblos Originarios
    "MAPUCHE": "1",
    "AYMARA": "1",
    "RAPA NUI (PASCUENSE)": "1",
    "DIAGUITA": "1",
    "KAWESQAR": "1", "KAWÃSQAR": "1", "KAWÉSQAR": "1",
    "QUECHUA": "1",
    "YAGAN (YAMANA)": "1", "YAGÁN (YÁMANA)": "1", "YAGÃN (YÃMANA)": "1",
    "COLLA": "1",
    "LICAN ANTAI (ATACAMENO)": "1", "LICAN ANTAI (ATACAMEÃO)": "1", "LICAN ANTAI (ATACAMEÑO)": "1",
}

# Nulos para ETNIA: 
nulos = ["NAN", "NONE", "NULL", "DESCONOCIDO", "NO RESPONDE"]

frecuencias, impactos = simular_transformacion_categorica(carpeta=carpeta_procesados, archivos=archivos_procesados, columna="ETNIA", mapeo=mapeo, valores_nulos=nulos)
mostrar_resultados(frecuencias, impactos, "ETNIA")


[ETNIA] Simulando mapeo y evaluando impacto de nulos...

Resultados para variable: ETNIA

Frecuencias por Cohorte:
- Cohorte Total:


,2019,2020,2021,2022,2023,2024
MAPPED,,,,,,
0,1130200,764157,798024,912499,1016486,1058851
1,20975,17597,18822,20263,23032,26831
NULOS_A_ELIMINAR,4,0,0,0,0,0


- Cohorte Oncológica:


,2019,2020,2021,2022,2023,2024
MAPPED,,,,,,
0,96420,63541,66648,76757,88963,96946
1,1400,1380,1278,1307,1751,2186


- Cohorte Control:


...

,Faltantes,MORT(0),MORT(1),SEV(0),SEV(1),SEV(2),SEV(3),CONS(0),CONS(1),CONS(2)
Año,,,,,,,,,,
2019,4,3,1,0,3,1,0,0,2,2
2020,0,0,0,0,0,0,0,0,0,0
2021,0,0,0,0,0,0,0,0,0,0
2022,0,0,0,0,0,0,0,0,0,0
2023,0,0,0,0,0,0,0,0,0,0
2024,0,0,0,0,0,0,0,0,0,0


- Cohorte Oncológica:


,Faltantes
Año,
2019,0
2020,0
2021,0
2022,0
2023,0
2024,0


- Cohorte Control:


,Faltantes,MORT(0),MORT(1),SEV(0),SEV(1),SEV(2),SEV(3),CONS(0),CONS(1),CONS(2)
Año,,,,,,,,,,
2019,4,3,1,0,3,1,0,0,2,2
2020,0,0,0,0,0,0,0,0,0,0
2021,0,0,0,0,0,0,0,0,0,0
2022,0,0,0,0,0,0,0,0,0,0
2023,0,0,0,0,0,0,0,0,0,0
2024,0,0,0,0,0,0,0,0,0,0


In [6]:
aplicar_transformacion_categorica(carpeta=carpeta_procesados, archivos=archivos_procesados, columna="ETNIA", mapeo=mapeo, valores_nulos=nulos, nueva_columna="ES_PUEBLO_ORIGINARIO")


[ETNIA] EJECUTANDO TRANSFORMACIÓN Y LIMPIEZA DEFINITIVA...
-> CREANDO NUEVA COLUMNA 'ES_PUEBLO_ORIGINARIO'
  -> 2019 | Inicial: 1,151,179 | Eliminadas: 4 | Final: 1,151,175
  -> 2020 | Inicial: 781,754 | Eliminadas: 0 | Final: 781,754
  -> 2021 | Inicial: 816,846 | Eliminadas: 0 | Final: 816,846
  -> 2022 | Inicial: 932,762 | Eliminadas: 0 | Final: 932,762
  -> 2023 | Inicial: 1,039,518 | Eliminadas: 0 | Final: 1,039,518
  -> 2024 | Inicial: 1,085,682 | Eliminadas: 0 | Final: 1,085,682
RESUMEN GLOBAL DE IMPACTO:
  Total filas analizadas: 5,807,741
  Total filas eliminadas: 4
  Total filas conservadas: 5,807,737


,2019,2020,2021,2022,2023,2024
ES_PUEBLO_ORIGINARIO,,,,,,
0,1130200,764157,798024,912499,1016486,1058851
1,20975,17597,18822,20263,23032,26831


### **2. Evaluación y Procesamiento de Variable Geográfica: Provincia (PROVINCIA)**
- **Objetivo de Simulación**: Evaluar la viabilidad de reducir la alta cardinalidad de la variable `PROVINCIA` (57 categorías) agrupándola en una nueva variable derivada correspondiente a las 16 Regiones de Chile (`REGION_RESIDENCIA`), y medir el impacto de la eliminación de registros sin ubicación geográfica válida.
- **Justificación de la Agrupación (Regiones vs. Macrozonas)**: La agrupación en 16 regiones representa el punto de equilibrio óptimo para evitar la "maldición de la dimensionalidad". Si bien una agrupación mayor (ej. 5 Macrozonas) reduciría aún más las columnas, destruiría la varianza espacial necesaria para caracterizar la desigualdad en la infraestructura hospitalaria (presencia de centros oncológicos de referencia, distancias de traslado y centralización). La tabla de frecuencias confirma que la distribución regional obtenida refleja fielmente la realidad demográfica del sistema de salud chileno.
- **Análisis de Impacto (Nulos)**: La etiqueta `DESCONOCIDO` demostró tener un impacto estadísticamente irrelevante. En la cohorte oncológica, la pérdida de datos se reduce a un máximo de 42 episodios en el año 2024, acumulando menos de 90 casos perdidos en todo el periodo 2019-2024 (sobre un universo cercano al medio millón de registros oncológicos). La tabla de impacto sobre los *targets* confirma que esta eliminación no sesga las clases minoritarias de Mortalidad (solo se pierden 6 casos de fallecidos en 2024) ni de Severidad.
- **Veredicto Final**: **Aprobado para Transformación y Eliminación de Nulos**. Se autoriza el uso del diccionario de mapeo para derivar la variable `REGION_RESIDENCIA`. Las filas con valor `DESCONOCIDO` se eliminarán del conjunto de datos, asegurando una caracterización territorial limpia y representativa.

In [7]:
# 1. Definir el diccionario de Mapeo (De 57 Provincias a 16 Regiones)
diccionario_provincias = {
    # XV - Arica y Parinacota
    "ARICA": "REGION_ARICA_PARINACOTA", "PARINACOTA": "REGION_ARICA_PARINACOTA",
    # I - Tarapacá
    "IQUIQUE": "REGION_TARAPACA","TAMARUGAL": "REGION_TARAPACA",
    # II - Antofagasta
    "ANTOFAGASTA": "REGION_ANTOFAGASTA", "EL LOA": "REGION_ANTOFAGASTA", "TOCOPILLA": "REGION_ANTOFAGASTA",
    # III - Atacama
    "COPIAPO": "REGION_ATACAMA", "HUASCO": "REGION_ATACAMA", "CHANARAL": "REGION_ATACAMA", "CHAÑARAL": "REGION_ATACAMA", "CHAÃARAL": "REGION_ATACAMA",
    # IV - Coquimbo
    "ELQUI": "REGION_COQUIMBO", "LIMARI": "REGION_COQUIMBO","CHOAPA": "REGION_COQUIMBO",
    # V - Valparaíso
    "VALPARAISO": "REGION_VALPARAISO", "MARGA MARGA": "REGION_VALPARAISO", "SAN ANTONIO": "REGION_VALPARAISO",
    "QUILLOTA": "REGION_VALPARAISO", "SAN FELIPE": "REGION_VALPARAISO", "LOS ANDES": "REGION_VALPARAISO",
    "PETORCA": "REGION_VALPARAISO", "ISLA DE PASCUA": "REGION_VALPARAISO",
    # RM - Metropolitana
    "SANTIAGO": "REGION_METROPOLITANA", "CORDILLERA": "REGION_METROPOLITANA", "MAIPO": "REGION_METROPOLITANA",
    "TALAGANTE": "REGION_METROPOLITANA", "MELIPILLA": "REGION_METROPOLITANA", "CHACABUCO": "REGION_METROPOLITANA",
    # VI - O'Higgins
    "CACHAPOAL": "REGION_OHIGGINS", "COLCHAGUA": "REGION_OHIGGINS", "CARDENAL CARO": "REGION_OHIGGINS",
    # VII - Maule
    "TALCA": "REGION_MAULE", "LINARES": "REGION_MAULE", "CURICO": "REGION_MAULE", "CAUQUENES": "REGION_MAULE",
    # XVI - Ñuble (Incluye la antigua provincia NUBLE y las actuales)
    "NUBLE": "REGION_NUBLE", "DIGUILLIN": "REGION_NUBLE", "PUNILLA": "REGION_NUBLE", "ITATA": "REGION_NUBLE",
    "DIGUILLÃN": "REGION_NUBLE", "DIGUILLÍN": "REGION_NUBLE", "ÃUBLE": "REGION_NUBLE",
    # VIII - Biobío
    "CONCEPCION": "REGION_BIOBIO", "BIO-BIO": "REGION_BIOBIO", "ARAUCO": "REGION_BIOBIO",
    # IX - La Araucanía
    "CAUTIN": "REGION_ARAUCANIA", "MALLECO": "REGION_ARAUCANIA",
    # XIV - Los Ríos
    "VALDIVIA": "REGION_LOS_RIOS", "RANCO": "REGION_LOS_RIOS",
    # X - Los Lagos
    "LLANQUIHUE": "REGION_LOS_LAGOS", "OSORNO": "REGION_LOS_LAGOS", "CHILOE": "REGION_LOS_LAGOS", "PALENA": "REGION_LOS_LAGOS",
    # XI - Aysén
    "COIHAIQUE": "REGION_AYSEN", "AISEN": "REGION_AYSEN", "GENERAL CARRERA": "REGION_AYSEN", "CAPITAN PRAT": "REGION_AYSEN",
    # XII - Magallanes y Antártica Chilena
    "MAGALLANES": "REGION_MAGALLANES", "ULTIMA ESPERANZA": "REGION_MAGALLANES", 
    "TIERRA DEL FUEGO": "REGION_MAGALLANES", "ANTARTICA CHILENA": "REGION_MAGALLANES", "ANTÁRTICA CHILENA": "REGION_MAGALLANES", "ANTÃRTICA CHILENA": "REGION_MAGALLANES"
}

# 2. Definir valores nulos
nulos_provincia = ["DESCONOCIDO", "NAN", "NONE", "NULL", ""]

# 3. Simular el impacto (Ojo con el nuevo parámetro para crear la variable derivada)
frecuencias, impactos = simular_transformacion_categorica(
    carpeta=carpeta_procesados, 
    archivos=archivos_procesados, 
    columna="PROVINCIA", 
    mapeo=diccionario_provincias, 
    valores_nulos=nulos_provincia
)

# 4. Mostrar resultados
mostrar_resultados(frecuencias, impactos, "PROVINCIA -> REGION_RESIDENCIA")


[PROVINCIA] Simulando mapeo y evaluando impacto de nulos...

Resultados para variable: PROVINCIA -> REGION_RESIDENCIA

Frecuencias por Cohorte:
- Cohorte Total:


,2019,2020,2021,2022,2023,2024
MAPPED,,,,,,
NULOS_A_ELIMINAR,808,623,40,43,280,393
REGION_ANTOFAGASTA,33472,27087,29163,31242,32576,34069
REGION_ARAUCANIA,77900,54129,54896,61137,67978,80512
REGION_ARICA_PARINACOTA,20482,15575,17109,18508,19980,20367
REGION_ATACAMA,21081,16200,17258,18689,20793,21661
REGION_AYSEN,10316,7754,7674,8708,10227,10541
REGION_BIOBIO,111000,73479,78282,93542,111692,118655
REGION_COQUIMBO,51795,34547,37724,43331,51739,52061
REGION_LOS_LAGOS,63651,43021,44602,50552,57380,59682


- Cohorte Oncológica:


,2019,2020,2021,2022,2023,2024
MAPPED,,,,,,
NULOS_A_ELIMINAR,21,17,0,3,20,42
REGION_ANTOFAGASTA,3318,2352,2181,2512,3063,3494
REGION_ARAUCANIA,5702,4522,4369,4603,5704,6804
REGION_ARICA_PARINACOTA,1766,1454,1376,1459,1761,1898
REGION_ATACAMA,1398,1246,1144,1213,1367,1552
REGION_AYSEN,522,512,486,550,542,634
REGION_BIOBIO,9123,6756,7644,9216,10687,11715
REGION_COQUIMBO,3479,2591,2987,3416,4159,4456
REGION_LOS_LAGOS,5530,3770,3752,3757,4633,5046


- Cohorte Control:


...

,Faltantes,MORT(0),MORT(1),SEV(0),SEV(1),SEV(2),SEV(3),CONS(0),CONS(1),CONS(2)
Año,,,,,,,,,,
2019,808,795,13,432,248,60,68,489,156,163
2020,623,595,28,100,285,78,160,282,149,192
2021,40,40,0,3,17,11,9,10,16,14
2022,43,41,2,4,23,6,10,16,16,11
2023,280,272,8,2,175,63,40,58,104,118
2024,393,376,17,10,256,66,61,107,145,141


- Cohorte Oncológica:


,Faltantes,MORT(0),MORT(1),SEV(0),SEV(1),SEV(2),SEV(3),CONS(0),CONS(1),CONS(2)
Año,,,,,,,,,,
2019,21,21,0,5,10,2,4,4,2,15
2020,17,17,0,0,10,5,2,1,11,5
2021,0,0,0,0,0,0,0,0,0,0
2022,3,3,0,1,1,1,0,2,1,0
2023,20,20,0,0,8,9,3,1,5,14
2024,42,36,6,0,25,8,9,2,10,30


- Cohorte Control:


,Faltantes,MORT(0),MORT(1),SEV(0),SEV(1),SEV(2),SEV(3),CONS(0),CONS(1),CONS(2)
Año,,,,,,,,,,
2019,787,774,13,427,238,58,64,485,154,148
2020,606,578,28,100,275,73,158,281,138,187
2021,40,40,0,3,17,11,9,10,16,14
2022,40,38,2,3,22,5,10,14,15,11
2023,260,252,8,2,167,54,37,57,99,104
2024,351,340,11,10,231,58,52,105,135,111


In [8]:
aplicar_transformacion_categorica(carpeta=carpeta_procesados, archivos=archivos_procesados, columna="PROVINCIA", mapeo=diccionario_provincias, 
                                  valores_nulos=nulos_provincia, nueva_columna="REGION")


[PROVINCIA] EJECUTANDO TRANSFORMACIÓN Y LIMPIEZA DEFINITIVA...
-> CREANDO NUEVA COLUMNA 'REGION'
  -> 2019 | Inicial: 1,151,175 | Eliminadas: 808 | Final: 1,150,367
  -> 2020 | Inicial: 781,754 | Eliminadas: 623 | Final: 781,131
  -> 2021 | Inicial: 816,846 | Eliminadas: 40 | Final: 816,806
  -> 2022 | Inicial: 932,762 | Eliminadas: 43 | Final: 932,719
  -> 2023 | Inicial: 1,039,518 | Eliminadas: 280 | Final: 1,039,238
  -> 2024 | Inicial: 1,085,682 | Eliminadas: 393 | Final: 1,085,289
RESUMEN GLOBAL DE IMPACTO:
  Total filas analizadas: 5,807,737
  Total filas eliminadas: 2,187
  Total filas conservadas: 5,805,550


,2019,2020,2021,2022,2023,2024
REGION,,,,,,
REGION_ANTOFAGASTA,33472,27087,29163,31242,32576,34069
REGION_ARAUCANIA,77900,54129,54896,61137,67978,80512
REGION_ARICA_PARINACOTA,20482,15575,17109,18508,19980,20367
REGION_ATACAMA,21081,16200,17258,18689,20793,21661
REGION_AYSEN,10316,7754,7674,8708,10227,10541
REGION_BIOBIO,111000,73479,78282,93542,111692,118655
REGION_COQUIMBO,51795,34547,37724,43331,51739,52061
REGION_LOS_LAGOS,63651,43021,44602,50552,57380,59682
REGION_LOS_RIOS,25128,16751,17165,19881,24514,23856


## Evaluación y Procesamiento de Variable Demográfica: Nacionalidad (NACIONALIDAD)

### 1. Descripción y Justificación
La variable `NACIONALIDAD` registra el país de origen o ciudadanía del paciente. El análisis exploratorio reveló desafíos significativos de dimensionalidad y densidad de datos:

* **Dimensionalidad Extrema y Errores de Registro:** La variable cuenta con más de 210 categorías únicas. Sin embargo, este volumen está artificialmente inflado por registros a nivel continental (ej. "AMERICA", "EUROPA", "AFRICA") y países con frecuencias marginales (1 a 10 casos por año).
* **Desbalance Estructural:** La categoría "CHILE" concentra sistemáticamente entre el 93% y el 97% de los episodios en todos los años. Aplicar técnicas de *One-Hot Encoding* sobre las 209 categorías restantes inyectaría una cantidad masiva de columnas vacías (matrices dispersas o *sparse matrices*), degradando el rendimiento de los algoritmos sin aportar varianza explicativa.
* **Valor Sociodemográfico:** A pesar del desbalance, la literatura epidemiológica señala que la condición de migrante suele correlacionarse con diagnósticos tardíos, barreras de acceso y diferencias en el consumo de recursos de alta complejidad. Por ende, la variable no debe ser descartada, sino sintetizada.

### 2. Estrategia de Transformación (Ingeniería de Características)
Se diseñó una estrategia de binarización automatizada para colapsar la extrema cardinalidad en una variable dicotómica de alto valor analítico:

* **Clase 0 (Nacional):** Asignada exclusivamente a la categoría `CHILE`.
* **Clase 1 (Extranjero):** Agrupación programática de las ~209 categorías restantes (países específicos y registros continentales). 
* **Automatización del Mapeo:** En lugar de codificar manualmente más de 200 llaves, se implementó un algoritmo en Python que extrae el *set* único de valores presentes en los 6 años y asigna dinámicamente el valor `1` a cualquier registro válido distinto de `CHILE`.

In [9]:
# 3. Mapeo para NACIONALIDAD
nacionalidades_unicas = set()
for archivo in archivos_procesados:
    df_temp = pd.read_csv(os.path.join(carpeta_procesados, archivo), usecols=["NACIONALIDAD"], low_memory=False)
    nacionalidades_unicas.update(df_temp["NACIONALIDAD"].dropna().str.strip().str.upper().unique())

diccionario_nacionalidad = {}
for pais in nacionalidades_unicas:
    if pais == "CHILE":
        diccionario_nacionalidad[pais] = "0"  # 0 = Línea base / Mayoría
    elif pais in ["DESCONOCIDO", "NAN", "NONE", "NULL", ""]:
        diccionario_nacionalidad[pais] = "DESCONOCIDO"
    else:
        diccionario_nacionalidad[pais] = "1"  # 1 = Rasgo de interés (Extranjero)

nulos_nacionalidad = ["DESCONOCIDO", "NAN", "NONE", "NULL", ""]

# Simular el impacto 
frecuencias, impactos = simular_transformacion_categorica(
    carpeta=carpeta_procesados, 
    archivos=archivos_procesados, 
    columna="NACIONALIDAD", 
    mapeo=diccionario_nacionalidad, 
    valores_nulos=nulos_nacionalidad
)

# 4. Mostrar resultados
mostrar_resultados(frecuencias, impactos, "NACIONALIDAD -> ES_EXTRANJERO")


[NACIONALIDAD] Simulando mapeo y evaluando impacto de nulos...

Resultados para variable: NACIONALIDAD -> ES_EXTRANJERO

Frecuencias por Cohorte:
- Cohorte Total:


,2019,2020,2021,2022,2023,2024
MAPPED,,,,,,
0,1085083,738471,769032,872533,975158,1019152
1,44694,39636,47774,60186,64080,66137
NULOS_A_ELIMINAR,20590,3024,0,0,0,0


- Cohorte Oncológica:


,2019,2020,2021,2022,2023,2024
MAPPED,,,,,,
0,95295,63292,66099,75694,87555,95407
1,1497,1454,1827,2367,3139,3683
NULOS_A_ELIMINAR,1007,158,0,0,0,0


- Cohorte Control:


...

,Faltantes,MORT(0),MORT(1),SEV(0),SEV(1),SEV(2),SEV(3),CONS(0),CONS(1),CONS(2)
Año,,,,,,,,,,
2019,20590,20044,546,4578,10014,4241,1757,8363,7279,4948
2020,3024,2887,137,89,1588,676,671,479,1265,1280
2021,0,0,0,0,0,0,0,0,0,0
2022,0,0,0,0,0,0,0,0,0,0
2023,0,0,0,0,0,0,0,0,0,0
2024,0,0,0,0,0,0,0,0,0,0


- Cohorte Oncológica:


,Faltantes,MORT(0),MORT(1),SEV(0),SEV(1),SEV(2),SEV(3),CONS(0),CONS(1),CONS(2)
Año,,,,,,,,,,
2019,1007,910,97,102,263,460,182,92,282,633
2020,158,129,29,9,36,52,61,14,67,77
2021,0,0,0,0,0,0,0,0,0,0
2022,0,0,0,0,0,0,0,0,0,0
2023,0,0,0,0,0,0,0,0,0,0
2024,0,0,0,0,0,0,0,0,0,0


- Cohorte Control:


,Faltantes,MORT(0),MORT(1),SEV(0),SEV(1),SEV(2),SEV(3),CONS(0),CONS(1),CONS(2)
Año,,,,,,,,,,
2019,19583,19134,449,4476,9751,3781,1575,8271,6997,4315
2020,2866,2758,108,80,1552,624,610,465,1198,1203
2021,0,0,0,0,0,0,0,0,0,0
2022,0,0,0,0,0,0,0,0,0,0
2023,0,0,0,0,0,0,0,0,0,0
2024,0,0,0,0,0,0,0,0,0,0


In [10]:
aplicar_transformacion_categorica(carpeta=carpeta_procesados, archivos=archivos_procesados, columna="NACIONALIDAD", 
                                  mapeo=diccionario_nacionalidad, valores_nulos=nulos_nacionalidad, nueva_columna="ES_EXTRANJERO")


[NACIONALIDAD] EJECUTANDO TRANSFORMACIÓN Y LIMPIEZA DEFINITIVA...
-> CREANDO NUEVA COLUMNA 'ES_EXTRANJERO'
  -> 2019 | Inicial: 1,150,367 | Eliminadas: 20,590 | Final: 1,129,777
  -> 2020 | Inicial: 781,131 | Eliminadas: 3,024 | Final: 778,107
  -> 2021 | Inicial: 816,806 | Eliminadas: 0 | Final: 816,806
  -> 2022 | Inicial: 932,719 | Eliminadas: 0 | Final: 932,719
  -> 2023 | Inicial: 1,039,238 | Eliminadas: 0 | Final: 1,039,238
  -> 2024 | Inicial: 1,085,289 | Eliminadas: 0 | Final: 1,085,289
RESUMEN GLOBAL DE IMPACTO:
  Total filas analizadas: 5,805,550
  Total filas eliminadas: 23,614
  Total filas conservadas: 5,781,936


,2019,2020,2021,2022,2023,2024
ES_EXTRANJERO,,,,,,
0,1085083,738471,769032,872533,975158,1019152
1,44694,39636,47774,60186,64080,66137


## Evaluación y Procesamiento de Variable Socioeconómica: Previsión (PREVISION)

### 1. Descripción y Justificación
La variable `PREVISION` indica el sistema de aseguramiento de salud del paciente. El análisis exploratorio identificó la necesidad de consolidar sus 14 categorías originales por dos motivos principales:

* **Fragmentación de la Libre Elección (FMLE):** El Fondo Nacional de Salud (FONASA) registra separadamente la Modalidad de Atención Institucional (MAI) y la Modalidad de Libre Elección (FMLE). Para propósitos de perfilamiento socioeconómico en un hospital público, ambas modalidades corresponden al mismo tramo de ingresos del paciente (B, C o D). Mantenerlas separadas diluye la representatividad de cada estrato.
* **Agrupación de Sistemas Menores:** Las Fuerzas Armadas y de Orden tienen tres cajas de previsión distintas (DIPRECA, CAPREDENA, SISA). Asimismo, los pacientes del sistema privado se dividen entre ISAPRE y PARTICULAR. Agrupar estos sistemas consolida las cohortes que históricamente representan una carga financiera distinta para la red pública, reduciendo el ruido estadístico.

### 2. Estrategia de Transformación (Ingeniería de Características)
Se diseñó un diccionario de mapeo para colapsar las 14 categorías en 6 macro-tramos estructurales, generando la variable derivada `TIPO_PREVISION`:

* **FONASA_A:** Personas carentes de recursos o migrantes sin regularizar (Copago 0%).
* **FONASA_B:** Personas con ingresos menores al sueldo mínimo (Copago 0%).
* **FONASA_C:** Personas con ingresos moderados (Copago 10%).
* **FONASA_D:** Personas con mayores ingresos dentro del sistema público (Copago 20%).
* **PRIVADO:** Agrupa ISAPRE y pacientes particulares que asumen el costo total o derivado de seguros privados.
* **FFAA_Y_ORDEN:** Agrupa DIPRECA, CAPREDENA y SISA (Sistemas con financiamiento estatal propio).

In [11]:
# 2. Mapeo para PREVISION
diccionario_prevision = {
    "FONASA INSTITUCIONAL - (MAI) A": "FONASA_A",
    "FONASA INSTITUCIONAL - (MAI) B": "FONASA_B",
    "FONASA INSTITUCIONAL - (MAI) C": "FONASA_C",
    "FONASA INSTITUCIONAL - (MAI) D": "FONASA_D",
    "FONASA LIBRE ELECCION (FMLE_B)": "FONASA_B",
    "FONASA LIBRE ELECCIÃN (FMLE_B)": "FONASA_B",
    "FONASA LIBRE ELECCIÓN (FMLE_B)": "FONASA_B",
    "FONASA LIBRE ELECCION (FMLE_C)": "FONASA_C",
    "FONASA LIBRE ELECCIÃN (FMLE_C)": "FONASA_C",
    "FONASA LIBRE ELECCIÓN (FMLE_C)": "FONASA_C",
    "FONASA LIBRE ELECCION (FMLE_D)": "FONASA_D",
    "FONASA LIBRE ELECCIÃN (FMLE_D)": "FONASA_D",
    "FONASA LIBRE ELECCIÓN (FMLE_D)": "FONASA_D",
    "PARTICULAR": "PRIVADO",
    "ISAPRE": "PRIVADO",
    "DIPRECA": "FFAA_Y_ORDEN",
    "CAPREDENA": "FFAA_Y_ORDEN",
    "CAJA DE PREVISION FFAA (SISA)": "FFAA_Y_ORDEN",
    "CAJA DE PREVISIÃN FFAA (SISA)": "FFAA_Y_ORDEN",
    "CAJA DE PREVISIÓN FFAA (SISA)": "FFAA_Y_ORDEN",
    "NO IDENTIFICADA": "DESCONOCIDO",
    "NO CONSIGNADO": "DESCONOCIDO"
}

nulos_prevision = ["DESCONOCIDO", "NAN", "NONE", "NULL", ""]

frecuencias, impactos = simular_transformacion_categorica(
    carpeta=carpeta_procesados, 
    archivos=archivos_procesados, 
    columna="PREVISION", 
    mapeo=diccionario_prevision, 
    valores_nulos=nulos_prevision
)
mostrar_resultados(frecuencias, impactos, "PREVISION -> TIPO_PREVISION")


[PREVISION] Simulando mapeo y evaluando impacto de nulos...

Resultados para variable: PREVISION -> TIPO_PREVISION

Frecuencias por Cohorte:
- Cohorte Total:


,2019,2020,2021,2022,2023,2024
MAPPED,,,,,,
FONASA_B,494725,351887,381821,448910,505691,538686
FONASA_A,303382,198405,195486,217650,229225,225811
FONASA_D,166036,114401,121948,135558,156135,165467
FONASA_C,132021,94897,100475,112058,132891,142012
PRIVADO,26178,14205,13669,15014,11614,9344
FFAA_Y_ORDEN,3927,3330,3398,3470,3658,3930
NULOS_A_ELIMINAR,3508,982,9,59,24,39


- Cohorte Oncológica:


,2019,2020,2021,2022,2023,2024
MAPPED,,,,,,
FFAA_Y_ORDEN,208,237,196,208,233,255
FONASA_A,18095,11859,11329,12460,13058,12591
FONASA_B,51190,35808,38323,45765,53709,59338
FONASA_C,10836,6936,7888,8443,10539,12292
FONASA_D,15594,9486,9847,10789,12808,14294
NULOS_A_ELIMINAR,297,43,4,6,0,3
PRIVADO,572,377,339,390,347,317


- Cohorte Control:


...

,Faltantes,MORT(0),MORT(1),SEV(0),SEV(1),SEV(2),SEV(3),CONS(0),CONS(1),CONS(2)
Año,,,,,,,,,,
2019,3508,3358,150,571,1944,545,448,1351,1007,1150
2020,982,923,59,28,557,183,214,395,256,331
2021,9,9,0,0,2,5,2,2,4,3
2022,59,56,3,6,22,17,14,16,20,23
2023,24,24,0,1,18,1,4,8,8,8
2024,39,37,2,3,19,5,12,7,15,17


- Cohorte Oncológica:


,Faltantes,MORT(0),MORT(1),SEV(0),SEV(1),SEV(2),SEV(3),CONS(0),CONS(1),CONS(2)
Año,,,,,,,,,,
2019,297,274,23,182,51,33,31,34,42,221
2020,43,40,3,3,6,21,13,4,17,22
2021,4,4,0,0,0,3,1,0,2,2
2022,6,5,1,0,1,3,2,0,1,5
2023,0,0,0,0,0,0,0,0,0,0
2024,3,3,0,0,1,1,1,1,2,0


- Cohorte Control:


,Faltantes,MORT(0),MORT(1),SEV(0),SEV(1),SEV(2),SEV(3),CONS(0),CONS(1),CONS(2)
Año,,,,,,,,,,
2019,3211,3084,127,389,1893,512,417,1317,965,929
2020,939,883,56,25,551,162,201,391,239,309
2021,5,5,0,0,2,2,1,2,2,1
2022,53,51,2,6,21,14,12,16,19,18
2023,24,24,0,1,18,1,4,8,8,8
2024,36,34,2,3,18,4,11,6,13,17


In [12]:
aplicar_transformacion_categorica(carpeta=carpeta_procesados, archivos=archivos_procesados, columna="PREVISION", 
                                  mapeo=diccionario_prevision, valores_nulos=nulos_prevision, nueva_columna="TIPO_PREVISION")


[PREVISION] EJECUTANDO TRANSFORMACIÓN Y LIMPIEZA DEFINITIVA...
-> CREANDO NUEVA COLUMNA 'TIPO_PREVISION'
  -> 2019 | Inicial: 1,129,777 | Eliminadas: 3,508 | Final: 1,126,269
  -> 2020 | Inicial: 778,107 | Eliminadas: 982 | Final: 777,125
  -> 2021 | Inicial: 816,806 | Eliminadas: 9 | Final: 816,797
  -> 2022 | Inicial: 932,719 | Eliminadas: 59 | Final: 932,660
  -> 2023 | Inicial: 1,039,238 | Eliminadas: 24 | Final: 1,039,214
  -> 2024 | Inicial: 1,085,289 | Eliminadas: 39 | Final: 1,085,250
RESUMEN GLOBAL DE IMPACTO:
  Total filas analizadas: 5,781,936
  Total filas eliminadas: 4,621
  Total filas conservadas: 5,777,315


,2019,2020,2021,2022,2023,2024
TIPO_PREVISION,,,,,,
FFAA_Y_ORDEN,3927,3330,3398,3470,3658,3930
FONASA_A,303382,198405,195486,217650,229225,225811
FONASA_B,494725,351887,381821,448910,505691,538686
FONASA_C,132021,94897,100475,112058,132891,142012
FONASA_D,166036,114401,121948,135558,156135,165467
PRIVADO,26178,14205,13669,15014,11614,9344


## Evaluación y Procesamiento de Variable Geográfico-Administrativa: Servicio de Salud (SERVICIO_SALUD)

### 1. Descripción y Justificación
La variable `SERVICIO_SALUD` identifica la red administrativa responsable de la gestión del episodio clínico. A diferencia de la provincia de residencia (que indica el origen demográfico del paciente), esta variable representa el flujo administrativo y la capacidad de resolución instalada a nivel local. El análisis exploratorio determinó la necesidad de una agrupación macrozonal por las siguientes razones:

* **Optimización de la Cardinalidad:** Reducir de 29 Servicios de Salud a 5 Macrozonas operativas permite que los modelos predictivos capturen patrones de gestión interregional sin diluir la potencia estadística en categorías de menor volumen.
* **Reflejo de la Red Asistencial Oncológica:** La estructura de Macrozonas (Norte, Centro, RM, Centro-Sur y Sur-Austral) coincide con la organización jerárquica del Ministerio de Salud (MINSAL) para la derivación de pacientes de alta complejidad. Los recursos oncológicos (como centros de radioterapia o unidades de trasplante de médula) se planifican y gestionan bajo esta lógica territorial, no a nivel provincial.
* **Complementariedad Geográfica (Prevención de Colinealidad):** Al agrupar los servicios en 5 macrozonas, se mantiene una variable de "gestión operativa" que no colisiona directamente con la granularidad de la variable `REGION_RESIDENCIA` (16 categorías). Esta doble resolución permite analizar el "Efecto Derivación" (pacientes cuyo origen regional difiere de su macrozona de tratamiento), factor crítico en el consumo de recursos.

### 2. Estrategia de Transformación (Ingeniería de Características)
Se aplicó un mapeo estructural para consolidar la variable original en la nueva categoría derivada `TIPO_SERVICIO_SALUD`:

* **MACROZONA_RM:** Consolida los 6 servicios de salud de la Región Metropolitana.
* **MACROZONA_NORTE:** Agrupa desde el Servicio de Salud Arica hasta Coquimbo.
* **MACROZONA_CENTRO:** Incluye Valparaíso-San Antonio, Viña del Mar-Quillota, Aconcagua, O'Higgins y Maule.
* **MACROZONA_CENTRO_SUR:** Consolida Ñuble, Concepción, Talcahuano, Biobío y las redes de Araucanía.
* **MACROZONA_SUR_AUSTRAL:** Agrupa desde Valdivia hasta Magallanes.

In [13]:
diccionario_servicio_salud = {
    # Macrozona Norte
    "ARICA": "MACROZONA_NORTE", "IQUIQUE": "MACROZONA_NORTE", 
    "ANTOFAGASTA": "MACROZONA_NORTE", "ATACAMA": "MACROZONA_NORTE", 
    "COQUIMBO": "MACROZONA_NORTE",
    
    # Macrozona Centro
    "VALPARAISO SAN ANTONIO": "MACROZONA_CENTRO", "VINA DEL MAR QUILLOTA": "MACROZONA_CENTRO", 
    "VIÃA DEL MAR QUILLOTA": "MACROZONA_CENTRO", "VIÑA DEL MAR QUILLOTA": "MACROZONA_CENTRO",
    "ACONCAGUA": "MACROZONA_CENTRO", "LIBERTADOR B. O HIGGINS": "MACROZONA_CENTRO", 
    "DEL MAULE": "MACROZONA_CENTRO",
    
    # Macrozona Metropolitana (Se mantiene separada por su altísimo volumen)
    "METROPOLITANO NORTE": "MACROZONA_RM", "METROPOLITANO OCCIDENTE": "MACROZONA_RM",
    "METROPOLITANO CENTRAL": "MACROZONA_RM", "METROPOLITANO ORIENTE": "MACROZONA_RM",
    "METROPOLITANO SUR": "MACROZONA_RM", "METROPOLITANO SURORIENTE": "MACROZONA_RM",
    
    # Macrozona Centro-Sur
    "NUBLE": "MACROZONA_CENTRO_SUR", "ÃUBLE": "MACROZONA_CENTRO_SUR", "ÑUBLE": "MACROZONA_CENTRO_SUR",
    "CONCEPCION": "MACROZONA_CENTRO_SUR", "CONCEPCIÃN": "MACROZONA_CENTRO_SUR", "CONCEPCIÓN": "MACROZONA_CENTRO_SUR",
    "TALCAHUANO": "MACROZONA_CENTRO_SUR", "BIOBIO": "MACROZONA_CENTRO_SUR",
    "ARAUCO": "MACROZONA_CENTRO_SUR", "ARAUCANIA NORTE": "MACROZONA_CENTRO_SUR",
    "ARAUCANÃA NORTE": "MACROZONA_CENTRO_SUR", "ARAUCANÍA NORTE": "MACROZONA_CENTRO_SUR",
    "ARAUCANIA SUR": "MACROZONA_CENTRO_SUR", "ARAUCANÃA SUR": "MACROZONA_CENTRO_SUR", "ARAUCANÍA SUR": "MACROZONA_CENTRO_SUR",
    
    # Macrozona Sur-Austral
    "VALDIVIA": "MACROZONA_SUR_AUSTRAL", "OSORNO": "MACROZONA_SUR_AUSTRAL",
    "DEL RELONCAVI": "MACROZONA_SUR_AUSTRAL", "DEL RELONCAVÃ": "MACROZONA_SUR_AUSTRAL", "DEL RELONCAVÍ": "MACROZONA_SUR_AUSTRAL",
    "CHILOE": "MACROZONA_SUR_AUSTRAL",
    "CHILOÃ": "MACROZONA_SUR_AUSTRAL", "CHILOÉ": "MACROZONA_SUR_AUSTRAL",
    "AYSEN": "MACROZONA_SUR_AUSTRAL", "MAGALLANES": "MACROZONA_SUR_AUSTRAL",
    
    # Nulos
    "DESCONOCIDO": "DESCONOCIDO", "NO APLICA": "DESCONOCIDO", "IGNORADO": "DESCONOCIDO"
}

nulos_servicio_salud = ["DESCONOCIDO", "NAN", "NONE", "NULL", ""]

frecuencias, impactos = simular_transformacion_categorica(
    carpeta=carpeta_procesados, 
    archivos=archivos_procesados, 
    columna="SERVICIO_SALUD", 
    mapeo=diccionario_servicio_salud, 
    valores_nulos=nulos_servicio_salud
)
mostrar_resultados(frecuencias, impactos, "SERVICIO_SALUD -> TIPO_SERVICIO_SALUD")


[SERVICIO_SALUD] Simulando mapeo y evaluando impacto de nulos...

Resultados para variable: SERVICIO_SALUD -> TIPO_SERVICIO_SALUD

Frecuencias por Cohorte:
- Cohorte Total:


,2019,2020,2021,2022,2023,2024
MAPPED,,,,,,
MACROZONA_RM,396298,275213,290179,329091,351865,354660
MACROZONA_CENTRO,251618,162413,166262,194185,220996,238104
MACROZONA_CENTRO_SUR,215751,152021,159900,186077,213352,234399
MACROZONA_NORTE,149701,110686,119758,131837,146461,149271
MACROZONA_SUR_AUSTRAL,112894,76542,80232,91414,105817,108249
NULOS_A_ELIMINAR,7,250,466,56,723,567


- Cohorte Oncológica:


,2019,2020,2021,2022,2023,2024
MAPPED,,,,,,
MACROZONA_CENTRO,24175,13534,13248,15874,18906,20770
MACROZONA_CENTRO_SUR,16664,13055,14107,16252,18912,21593
MACROZONA_NORTE,11380,8868,8968,10042,11969,13402
MACROZONA_RM,34735,22259,24510,28298,31866,33602
MACROZONA_SUR_AUSTRAL,9541,6970,7071,7587,9008,9686
NULOS_A_ELIMINAR,0,17,18,2,33,34


- Cohorte Control:


...

,Faltantes,MORT(0),MORT(1),SEV(0),SEV(1),SEV(2),SEV(3),CONS(0),CONS(1),CONS(2)
Año,,,,,,,,,,
2019,7,7,0,0,4,3,0,2,3,2
2020,250,246,4,4,142,49,55,88,79,83
2021,466,442,24,9,281,76,100,198,136,132
2022,56,55,1,8,27,9,12,21,18,17
2023,723,707,16,11,421,162,129,212,248,263
2024,567,550,17,9,355,110,93,163,161,243


- Cohorte Oncológica:


,Faltantes,MORT(0),MORT(1),SEV(0),SEV(1),SEV(2),SEV(3),CONS(0),CONS(1),CONS(2)
Año,,,,,,,,,,
2019,0,0,0,0,0,0,0,0,0,0
2020,17,17,0,0,7,7,3,0,7,10
2021,18,15,3,1,6,5,6,0,9,9
2022,2,2,0,0,1,1,0,0,2,0
2023,33,31,2,1,7,12,13,0,8,25
2024,34,31,3,1,19,7,7,5,10,19


- Cohorte Control:


,Faltantes,MORT(0),MORT(1),SEV(0),SEV(1),SEV(2),SEV(3),CONS(0),CONS(1),CONS(2)
Año,,,,,,,,,,
2019,7,7,0,0,4,3,0,2,3,2
2020,233,229,4,4,135,42,52,88,72,73
2021,448,427,21,8,275,71,94,198,127,123
2022,54,53,1,8,26,8,12,21,16,17
2023,690,676,14,10,414,150,116,212,240,238
2024,533,519,14,8,336,103,86,158,151,224


In [14]:
aplicar_transformacion_categorica(carpeta=carpeta_procesados, archivos=archivos_procesados, columna="SERVICIO_SALUD", 
                                  mapeo=diccionario_servicio_salud, valores_nulos=nulos_servicio_salud, nueva_columna="TIPO_SERVICIO_SALUD")


[SERVICIO_SALUD] EJECUTANDO TRANSFORMACIÓN Y LIMPIEZA DEFINITIVA...
-> CREANDO NUEVA COLUMNA 'TIPO_SERVICIO_SALUD'
  -> 2019 | Inicial: 1,126,269 | Eliminadas: 7 | Final: 1,126,262
  -> 2020 | Inicial: 777,125 | Eliminadas: 250 | Final: 776,875
  -> 2021 | Inicial: 816,797 | Eliminadas: 466 | Final: 816,331
  -> 2022 | Inicial: 932,660 | Eliminadas: 56 | Final: 932,604
  -> 2023 | Inicial: 1,039,214 | Eliminadas: 723 | Final: 1,038,491
  -> 2024 | Inicial: 1,085,250 | Eliminadas: 567 | Final: 1,084,683
RESUMEN GLOBAL DE IMPACTO:
  Total filas analizadas: 5,777,315
  Total filas eliminadas: 2,069
  Total filas conservadas: 5,775,246


,2019,2020,2021,2022,2023,2024
TIPO_SERVICIO_SALUD,,,,,,
MACROZONA_CENTRO,251618,162413,166262,194185,220996,238104
MACROZONA_CENTRO_SUR,215751,152021,159900,186077,213352,234399
MACROZONA_NORTE,149701,110686,119758,131837,146461,149271
MACROZONA_RM,396298,275213,290179,329091,351865,354660
MACROZONA_SUR_AUSTRAL,112894,76542,80232,91414,105817,108249


## Evaluación y Procesamiento de Variable Clínica Operativa: Tipo de Procedencia (TIPO_PROCEDENCIA)

### 1. Descripción y Justificación
La variable `TIPO_PROCEDENCIA` describe el origen clínico o físico del paciente antes del ingreso al hospital. Su análisis exploratorio reveló fragmentación y quiebres temporales que requerían normalización:

* **Fragmentación de la Red Pública:** Los ingresos ambulatorios y de urgencia estaban hiper-segmentados (ej. SAPU, SUR, SUC, Posta Rural). Agruparlos por nivel de complejidad (Atención Primaria vs. Nivel Secundario/Terciario) estabiliza la varianza.
* **Quiebre Temporal por Programas Ministeriales:** Categorías como *Plan de Resolución LE* o *Estrategia CRR* registran cero ingresos antes de 2022. Estas etiquetas administrativas fueron creadas por el MINSAL para financiar la reducción de listas de espera post-pandemia. Clínicamente, son derivaciones electivas. Para evitar que los modelos asimilen erróneamente que "apareció" un nuevo tipo de paciente en 2022, estas etiquetas se fusionaron con la categoría genérica de derivación de nivel secundario.
* **Población Institucionalizada:** Pacientes provenientes de cárceles, SENAME o ELEAM poseen un perfil de vulnerabilidad social y riesgo biomédico único. Separarlos en la categoría `INSTITUCION_CERRADA` evita enmascararlos dentro de "Otras Instituciones".
* **Corrupción de Encoding:** Se detectaron múltiples variables duplicadas por fallos de codificación (ej. `HOSPITALIZACIÃ“N DIURNA`), las cuales fueron absorbidas por el diccionario de mapeo.

### 2. Estrategia de Transformación (Ingeniería de Características)
Se colapsaron las 20 subcategorías en 7 flujos estructurales de ingreso, creando la variable derivada `TIPO_PROCEDENCIA_AGRUPADA`:

* **URGENCIA_DOMICILIO:** Consulta espontánea en el servicio de emergencia hospitalario.
* **URGENCIA_RED_PRIMARIA:** Derivados desde la urgencia de atención primaria (SAPU, SUR, Postas Rurales).
* **AMBULATORIO_ESPECIALIDAD:** Derivación electiva y resoluciones de lista de espera (CDT, CRS, CMA, Hospitalización Domiciliaria/Diurna, Estrategia CRR, Pago GRD).
* **AMBULATORIO_APS:** Derivación electiva directa desde la Atención Primaria (CESFAM).
* **TRASLADO_RED_PUBLICA:** Traslados desde otros hospitales estatales o por la Unidad de Gestión Centralizada de Camas (UGCC).
* **TRASLADO_PRIVADO:** Traslados y derivaciones desde consultas o clínicas privadas.
* **INSTITUCION_CERRADA:** Población penal o institucionalizada.

### 3. Análisis de Resultados de la Simulación

**Impacto en Variables Objetivo (Nulos):**
Los casos nulos o etiquetados como "No Identificado" (`DESCONOCIDO`) presentan frecuencias insignificantes en la cohorte oncológica. El único *peak* anómalo ocurrió en 2021, pero representó la pérdida de apenas **33 episodios oncológicos**. En todos los demás años, la pérdida osciló entre 0 y 28 pacientes.

**Veredicto Final:** **Aprobado para Transformación y Eliminación de Nulos**. La agrupación resuelve la inconsistencia temporal de los programas post-pandemia y robustece las señales de derivación. La eliminación de nulos está plenamente justificada al no tener impacto estadístico en la distribución de la Severidad ni de la Mortalidad oncológica.

In [ ]:
# =====================================================================
# TIPO_PROCEDENCIA (Corregido: Consistencia Temporal y Encoding)
# =====================================================================
diccionario_procedencia = {
    # Urgencia / Espontánea
    "SERVICIO EMERGENCIA (DOMICILIO)": "URGENCIA_DOMICILIO",
    "APS URGENCIA (SAPU, SUR, SUC)": "URGENCIA_RED_PRIMARIA",
    "POSTA RURAL": "URGENCIA_RED_PRIMARIA",
    
    # Derivación Electiva / Ambulatoria (Absorbe programas de resolución para evitar quiebre temporal 2022)
    "CENTRO ESPECIALIDADES (CDT, CRS, CONSULTORIO ADOS. ESP)": "AMBULATORIO_ESPECIALIDAD",
    "CIRUGIA MAYOR AMBULATORIA (CMA)": "AMBULATORIO_ESPECIALIDAD", 
    "CIRUGÃA MAYOR AMBULATORIA (CMA)": "AMBULATORIO_ESPECIALIDAD",
    "CIRUGÍA MAYOR AMBULATORIA (CMA)": "AMBULATORIO_ESPECIALIDAD",
    "HOSPITALIZACION DIURNA": "AMBULATORIO_ESPECIALIDAD", 
    "HOSPITALIZACIÃN DIURNA": "AMBULATORIO_ESPECIALIDAD",
    "HOSPITALIZACIÓN DIURNA": "AMBULATORIO_ESPECIALIDAD",
    "HOSPITALIZACION DOMICILIARIA": "AMBULATORIO_ESPECIALIDAD",
    "HOSPITALIZACIÃN DOMICILIARIA": "AMBULATORIO_ESPECIALIDAD",
    "HOSPITALIZACIÓN DOMICILIARIA": "AMBULATORIO_ESPECIALIDAD",
    "PLAN DE RESOLUCION LE": "AMBULATORIO_ESPECIALIDAD",
    "PLAN DE RESOLUCIÃ“N LE": "AMBULATORIO_ESPECIALIDAD",
    "PLAN DE RESOLUCIÓN LE": "AMBULATORIO_ESPECIALIDAD",
    "ESTRATEGIA CRR": "AMBULATORIO_ESPECIALIDAD",
    "LISTA DE ESPERA": "AMBULATORIO_ESPECIALIDAD",
    "CARDIOCIRUGIA PAGO GRD": "AMBULATORIO_ESPECIALIDAD", 
    "CARDIOCIRUGÃA PAGO GRD": "AMBULATORIO_ESPECIALIDAD",
    "CARDIOCIRUGÍA PAGO GRD": "AMBULATORIO_ESPECIALIDAD",
    
    # Atención Primaria
    "APS CONSULTORIO (CESFAM)": "AMBULATORIO_APS",
    
    # Traslados Red Pública
    "OTROS HOSPITALES DE LA RED": "TRASLADO_RED_PUBLICA",
    "OTROS HOSPITALES RED NACIONAL": "TRASLADO_RED_PUBLICA",
    "UGCC": "TRASLADO_RED_PUBLICA",
    
    # Traslados Privados
    "CONSULTA PRIVADA": "TRASLADO_PRIVADO",
    "OTRAS INSTITUCIONES SALUD (CLINICAS PRIVADAS, DE REHABILITAC": "TRASLADO_PRIVADO",
    "OTRAS INSTITUCIONES SALUD (CLÃNICAS PRIVADAS, DE REHABILITAC": "TRASLADO_PRIVADO",
    "OTRAS INSTITUCIONES SALUD (CLÍNICAS PRIVADAS, DE REHABILITAC": "TRASLADO_PRIVADO",
    
    # Instituciones Cerradas / Cautivas
    "OTRAS INSTITUCIONES (CARCEL, HOGARES DE ANCIANOS, SENAME, EC": "INSTITUCION_CERRADA",
    "OTRAS INSTITUCIONES (CÃRCEL, HOGARES DE ANCIANOS, SENAME, EC": "INSTITUCION_CERRADA",
    "OTRAS INSTITUCIONES (CÁRCEL, HOGARES DE ANCIANOS, SENAME, EC": "INSTITUCION_CERRADA",
    
    # Nulos
    "DESCONOCIDO": "DESCONOCIDO",
    "NO IDENTIFICADO": "DESCONOCIDO"
}

nulos_procedencia = ["DESCONOCIDO", "NAN", "NONE", "NULL", ""]

frecuencias, impactos = simular_transformacion_categorica(
    carpeta=carpeta_procesados, 
    archivos=archivos_procesados, 
    columna="TIPO_PROCEDENCIA", 
    mapeo=diccionario_procedencia, 
    valores_nulos=nulos_procedencia
)
mostrar_resultados(frecuencias, impactos, "TIPO_PROCEDENCIA") 
# AMBULATORIO_APS, AMBULATORIO_ESPECIALIDAD, INSTITUCION_CERRADA, TRASLADO_PRIVADO, TRASLADO_RED_PUBLICA, URGENCIA_DOMICILIO, URGENCIA_RED_PRIMARIA


[TIPO_PROCEDENCIA] Simulando mapeo y evaluando impacto de nulos...

Resultados para variable: TIPO_PROCEDENCIA

Frecuencias por Cohorte:
- Cohorte Total:


,2019,2020,2021,2022,2023,2024
MAPPED,,,,,,
AMBULATORIO_APS,19524,15186,13787,12890,14574,15405
AMBULATORIO_ESPECIALIDAD,410354,211810,234831,326554,384507,404451
INSTITUCION_CERRADA,2996,3821,5857,3380,5501,6785
NULOS_A_ELIMINAR,207,95,3089,2,2,5
TRASLADO_PRIVADO,50965,27490,27855,33849,34872,36055
TRASLADO_RED_PUBLICA,82198,73360,75379,74186,83044,89630
URGENCIA_DOMICILIO,521486,407285,408883,436643,464421,474573
URGENCIA_RED_PRIMARIA,38532,37828,46650,45100,51570,57779


- Cohorte Oncológica:


,2019,2020,2021,2022,2023,2024
MAPPED,,,,,,
AMBULATORIO_APS,771,678,576,566,681,777
AMBULATORIO_ESPECIALIDAD,62229,35175,38613,45780,52133,56381
INSTITUCION_CERRADA,101,270,234,181,606,849
NULOS_A_ELIMINAR,28,26,33,0,0,0
TRASLADO_PRIVADO,2049,1403,1378,1513,1761,1846
TRASLADO_RED_PUBLICA,4640,4293,4070,4310,5318,6195
URGENCIA_DOMICILIO,25127,21207,21100,23545,27702,29888
URGENCIA_RED_PRIMARIA,1550,1634,1900,2158,2460,3117


- Cohorte Control:


...

,Faltantes,MORT(0),MORT(1),SEV(0),SEV(1),SEV(2),SEV(3),CONS(0),CONS(1),CONS(2)
Año,,,,,,,,,,
2019,207,201,6,6,141,32,28,50,99,58
2020,95,89,6,0,55,21,19,27,32,36
2021,3089,3087,2,2109,676,273,31,1397,1279,413
2022,2,2,0,0,1,0,1,1,0,1
2023,2,1,1,0,1,0,1,1,0,1
2024,5,5,0,0,3,1,1,2,1,2


- Cohorte Oncológica:


,Faltantes,MORT(0),MORT(1),SEV(0),SEV(1),SEV(2),SEV(3),CONS(0),CONS(1),CONS(2)
Año,,,,,,,,,,
2019,28,28,0,0,7,5,16,2,13,13
2020,26,26,0,0,11,4,11,1,6,19
2021,33,33,0,14,8,9,2,9,13,11
2022,0,0,0,0,0,0,0,0,0,0
2023,0,0,0,0,0,0,0,0,0,0
2024,0,0,0,0,0,0,0,0,0,0


- Cohorte Control:


,Faltantes,MORT(0),MORT(1),SEV(0),SEV(1),SEV(2),SEV(3),CONS(0),CONS(1),CONS(2)
Año,,,,,,,,,,
2019,179,173,6,6,134,27,12,48,86,45
2020,69,63,6,0,44,17,8,26,26,17
2021,3056,3054,2,2095,668,264,29,1388,1266,402
2022,2,2,0,0,1,0,1,1,0,1
2023,2,1,1,0,1,0,1,1,0,1
2024,5,5,0,0,3,1,1,2,1,2


In [16]:
aplicar_transformacion_categorica(carpeta=carpeta_procesados, archivos=archivos_procesados, columna="TIPO_PROCEDENCIA", 
                                  mapeo=diccionario_procedencia, valores_nulos=nulos_procedencia)


[TIPO_PROCEDENCIA] EJECUTANDO TRANSFORMACIÓN Y LIMPIEZA DEFINITIVA...
-> SOBRESCRIBIENDO 'TIPO_PROCEDENCIA'
  -> 2019 | Inicial: 1,126,262 | Eliminadas: 207 | Final: 1,126,055
  -> 2020 | Inicial: 776,875 | Eliminadas: 95 | Final: 776,780
  -> 2021 | Inicial: 816,331 | Eliminadas: 3,089 | Final: 813,242
  -> 2022 | Inicial: 932,604 | Eliminadas: 2 | Final: 932,602
  -> 2023 | Inicial: 1,038,491 | Eliminadas: 2 | Final: 1,038,489
  -> 2024 | Inicial: 1,084,683 | Eliminadas: 5 | Final: 1,084,678
RESUMEN GLOBAL DE IMPACTO:
  Total filas analizadas: 5,775,246
  Total filas eliminadas: 3,400
  Total filas conservadas: 5,771,846


,2019,2020,2021,2022,2023,2024
TIPO_PROCEDENCIA,,,,,,
AMBULATORIO_APS,19524,15186,13787,12890,14574,15405
AMBULATORIO_ESPECIALIDAD,410354,211810,234831,326554,384507,404451
INSTITUCION_CERRADA,2996,3821,5857,3380,5501,6785
TRASLADO_PRIVADO,50965,27490,27855,33849,34872,36055
TRASLADO_RED_PUBLICA,82198,73360,75379,74186,83044,89630
URGENCIA_DOMICILIO,521486,407285,408883,436643,464421,474573
URGENCIA_RED_PRIMARIA,38532,37828,46650,45100,51570,57779


## Evaluación y Procesamiento de Variable Clínica Operativa: Tipo de Ingreso (TIPO_INGRESO)

### 1. Descripción y Justificación
La variable `TIPO_INGRESO` clasifica la naturaleza de la admisión del paciente al recinto hospitalario. El análisis de esta variable demostró que requería un procesamiento mínimo, pero crítico para garantizar la coherencia de los datos:

* **Categorización Estable:** La base de datos original clasifica la inmensa mayoría de los ingresos en tres grandes flujos operativos: Urgencia, Programada y Obstétrica. Esta categorización es clínicamente robusta y no requiere agrupación compleja.
* **Inconsistencia Semántica:** Se detectó la categoría minoritaria `NO PROGRAMADA`, la cual, en términos de gestión de camas y flujos clínicos, es sinónimo de un ingreso no electivo o de urgencia.
* **Manejo de Nulos:** Las categorías `NO IDENTIFICADA` y `DESCONOCIDO` representaban ruido estadístico que debía ser evaluado para su potencial eliminación.

### 2. Estrategia de Transformación (Ingeniería de Características)
Se aplicó un mapeo directo para consolidar la semántica de la variable y estandarizar la nomenclatura, generando la variable derivada (que sobrescribe la original):

* **URGENCIA:** Absorbe los casos originalmente etiquetados como `URGENCIA` y `NO PROGRAMADA`.
* **ELECTIVO:** Renombra la categoría original `PROGRAMADA` para utilizar nomenclatura clínica estándar.
* **OBSTETRICO:** Se mantiene la categoría original `OBSTETRICA`.

### 3. Análisis de Resultados de la Simulación

**Distribución de Frecuencias:**
La distribución refleja con gran exactitud la realidad asistencial oncológica chilena. Para el año 2024, el mayor volumen de pacientes ingresa por vía **ELECTIVA** (~55.600 casos), lo que refleja el tratamiento programado (cirugías, quimioterapias, etc.). Sin embargo, existe un volumen alarmante de pacientes que ingresan por vía de **URGENCIA** (~43.000 casos), lo cual es un indicador clásico de descompensación de la patología de base o toxicidad del tratamiento. Los ingresos **OBSTÉTRICOS** en la cohorte oncológica son, como es de esperar, anecdóticos (~370 casos en 2024), reafirmando hallazgos previos.

**Impacto en Variables Objetivo (Nulos):**
La etiqueta `NULOS_A_ELIMINAR` (que agrupa `DESCONOCIDO` y `NO IDENTIFICADA`) demostró tener un impacto estadísticamente irrelevante.

* **Cohorte Oncológica:** El volumen de nulos osciló entre **0 y 9 casos anuales** durante todo el sexenio analizado.
* **Preservación de Targets:** La eliminación de estos escasos registros no afectó la variable de Mortalidad (cero pacientes fallecidos perdidos en todos los años) ni alteró las distribuciones de Severidad y Consumo.
* **Veredicto Final:** **Aprobado para Transformación y Eliminación de Nulos**. La limpieza de estas categorías erráticas asegura la consistencia de los datos sin introducir sesgos de selección, permitiendo que el modelo confíe en la naturaleza del ingreso como predictor de severidad.

In [17]:
# =====================================================================
# 3. TIPO_INGRESO (Limpieza y Binarización Electivo/Urgencia)
# =====================================================================
diccionario_ingreso = {
    "URGENCIA": "URGENCIA",
    "NO PROGRAMADA": "URGENCIA",
    "PROGRAMADA": "ELECTIVO",
    "OBSTETRICA": "OBSTETRICO",
    "NO IDENTIFICADA": "DESCONOCIDO",
    "DESCONOCIDO": "DESCONOCIDO"
}

nulos_ingreso = ["DESCONOCIDO", "NAN", "NONE", "NULL", ""]

frecuencias, impactos = simular_transformacion_categorica(
    carpeta=carpeta_procesados, 
    archivos=archivos_procesados, 
    columna="TIPO_INGRESO", 
    mapeo=diccionario_ingreso, 
    valores_nulos=nulos_ingreso
)
mostrar_resultados(frecuencias, impactos, "TIPO_INGRESO") 


[TIPO_INGRESO] Simulando mapeo y evaluando impacto de nulos...

Resultados para variable: TIPO_INGRESO

Frecuencias por Cohorte:
- Cohorte Total:


,2019,2020,2021,2022,2023,2024
MAPPED,,,,,,
URGENCIA,535233,428127,457189,468388,517930,551432
ELECTIVO,412133,198715,217679,307301,366088,388375
OBSTETRICO,178676,149920,138373,156880,154416,144828
NULOS_A_ELIMINAR,13,18,1,33,55,43


- Cohorte Oncológica:


,2019,2020,2021,2022,2023,2024
MAPPED,,,,,,
ELECTIVO,63276,36340,38143,45074,52182,55685
NULOS_A_ELIMINAR,2,0,0,5,1,9
OBSTETRICO,192,218,199,301,305,371
URGENCIA,32997,28102,29529,32673,38173,42988


- Cohorte Control:


...

,Faltantes,MORT(0),MORT(1),SEV(0),SEV(1),SEV(2),SEV(3),CONS(0),CONS(1),CONS(2)
Año,,,,,,,,,,
2019,13,12,1,2,7,4,0,4,3,6
2020,18,18,0,2,8,5,3,4,6,8
2021,1,1,0,0,0,1,0,0,1,0
2022,33,33,0,0,12,13,8,6,14,13
2023,55,55,0,0,28,18,9,14,28,13
2024,43,43,0,0,16,19,8,10,17,16


- Cohorte Oncológica:


,Faltantes,MORT(0),MORT(1),SEV(0),SEV(1),SEV(2),SEV(3),CONS(0),CONS(1),CONS(2)
Año,,,,,,,,,,
2019,2,2,0,0,0,2,0,0,0,2
2020,0,0,0,0,0,0,0,0,0,0
2021,0,0,0,0,0,0,0,0,0,0
2022,5,5,0,0,1,2,2,0,1,4
2023,1,1,0,0,0,0,1,0,0,1
2024,9,9,0,0,2,5,2,0,5,4


- Cohorte Control:


,Faltantes,MORT(0),MORT(1),SEV(0),SEV(1),SEV(2),SEV(3),CONS(0),CONS(1),CONS(2)
Año,,,,,,,,,,
2019,11,10,1,2,7,2,0,4,3,4
2020,18,18,0,2,8,5,3,4,6,8
2021,1,1,0,0,0,1,0,0,1,0
2022,28,28,0,0,11,11,6,6,13,9
2023,54,54,0,0,28,18,8,14,28,12
2024,34,34,0,0,14,14,6,10,12,12


In [18]:
aplicar_transformacion_categorica(carpeta=carpeta_procesados, archivos=archivos_procesados, columna="TIPO_INGRESO", 
                                  mapeo=diccionario_ingreso, valores_nulos=nulos_ingreso)


[TIPO_INGRESO] EJECUTANDO TRANSFORMACIÓN Y LIMPIEZA DEFINITIVA...
-> SOBRESCRIBIENDO 'TIPO_INGRESO'
  -> 2019 | Inicial: 1,126,055 | Eliminadas: 13 | Final: 1,126,042
  -> 2020 | Inicial: 776,780 | Eliminadas: 18 | Final: 776,762
  -> 2021 | Inicial: 813,242 | Eliminadas: 1 | Final: 813,241
  -> 2022 | Inicial: 932,602 | Eliminadas: 33 | Final: 932,569
  -> 2023 | Inicial: 1,038,489 | Eliminadas: 55 | Final: 1,038,434
  -> 2024 | Inicial: 1,084,678 | Eliminadas: 43 | Final: 1,084,635
RESUMEN GLOBAL DE IMPACTO:
  Total filas analizadas: 5,771,846
  Total filas eliminadas: 163
  Total filas conservadas: 5,771,683


,2019,2020,2021,2022,2023,2024
TIPO_INGRESO,,,,,,
ELECTIVO,412133,198715,217679,307301,366088,388375
OBSTETRICO,178676,149920,138373,156880,154416,144828
URGENCIA,535233,428127,457189,468388,517930,551432


## Evaluación y Procesamiento de Variable Clínica: Especialidad Médica (ESPECIALIDAD_MEDICA)

### 1. Descripción y Justificación
La variable `ESPECIALIDAD_MEDICA` identifica el área clínica responsable de la gestión del paciente. Su auditoría reveló tres problemas estructurales incompatibles con el modelado predictivo (GRD):

* **Cardinalidad Extrema (Micro-clases):** La base contenía más de 103 categorías originales. Docenas de ellas (ej. Genética Clínica, Medicina del Deporte, Nutriólogo) aportaban menos de 10 casos anuales. Mantenerlas generaría matrices fuertemente desbalanceadas, induciendo *overfitting* al obligar al modelo a encontrar patrones donde no hay significancia estadística.
* **Corrupción Tipográfica (Mojibake):** Se detectó que las exportaciones del MINSAL pre-2022 presentaban fallas de codificación (ej. *CIRUGÍA* registrado como *CIRUGÃA*). Un mapeo tradicional basado en coincidencias exactas fallaba, enviando cientos de miles de registros a categorías "basurero" o fragmentando áreas idénticas.
* **Agrupación Semántica del Consumo:** Clínicamente, especialidades derivadas (ej. Geriatría, Reumatología) comparten perfiles de consumo hospitalario casi idénticos con su especialidad troncal (Medicina Interna), ya que utilizan la misma infraestructura de sala común y no requieren derechos de pabellón.

### 2. Estrategia de Transformación (Ingeniería de Características)
Para resolver la fragmentación y el error de codificación, se diseñó un algoritmo de **extracción de raíces (*stemming*)** resistente a variaciones tipográficas (ej. buscar `"CIRUG"` en lugar de la palabra completa). El modelo colapsó las 124 variaciones de texto en **19 Macro-Áreas Clínicas**:

* **Flujos Base:**
    * **CIRUGIA_GENERAL** y **ESPECIALIDADES_QUIRURGICAS:** Agrupan todo el flujo de pabellón (incluyendo apoyo anestésico).
    * **MEDICINA_INTERNA_Y_SUBS:** Absorbe la sala médica general y micro-subespecialidades de perfil de consumo homólogo (Dermatología, Endocrinología, Geriatría, Reumatología, Rehabilitación e Inmunología/Infectología).
    * **PEDIATRIA** y **GINECO_OBSTETRICIA:** Consolidan los flujos materno-infantiles.
* **Flujos de Alta Complejidad:**
    * **CUIDADOS_CRITICOS** y **URGENCIA:** Se separaron obligatoriamente debido a su perfil de consumo diametralmente opuesto (estabilización rápida vs. soporte vital intensivo).
    * **ONCOLOGIA:** Aísla las camas exclusivas de tratamiento del cáncer y radioterapia.
* **Especialidades Médicas de Alto Volumen:** Se mantuvieron independientes aquellas especialidades médicas con alto impacto en el sistema GRD (Cardiovascular, Respiratorio, Renal, Neurología, Psiquiatría y Hematología).
* **Apoyo:** Se unificaron Imagenología y Laboratorio en `APOYO_CLINICO_DIAGNOSTICO`.

### 3. Análisis de Resultados de la Simulación

**Distribución de Frecuencias:**
La transformación optimizó la distribución, eliminando el "ruido" estadístico sin perder sensibilidad clínica. En la cohorte oncológica (2024), destacan las áreas **ESPECIALIDADES_QUIRURGICAS** (~23.700 casos) y **CIRUGIA_GENERAL** (~22.200 casos), reafirmando que el manejo del cáncer es inminentemente quirúrgico. El área de **MEDICINA_INTERNA_Y_SUBS** mantiene una frecuencia alta (~20.400 casos) que refleja fielmente el volumen de pacientes oncológicos tratados por descompensaciones generales sin indicación de cirugía.

**Impacto en Variables Objetivo (Nulos):**
La eliminación de la etiqueta `NULOS_A_ELIMINAR` (registros vacíos o "Desconocidos") evidenció un impacto nulo en el dataset reciente:

* **Calidad Perfecta Post-2021:** Desde 2021 a la fecha, la pérdida de nulos es de exactamente **cero** casos en todas las cohortes.
* **Ausencia de Sesgo:** La eliminación histórica se restringe únicamente al año 2019, con apenas **20 pacientes oncológicos** perdidos, representando menos del **0.02%** de la base anual.
* **Veredicto Final:** **Aprobado para Transformación y Eliminación de Nulos**. El mapeo por raíces (*stemming*) recuperó la integridad de la variable, consolidando 19 macro-categorías estadísticamente densas y clínicamente diferenciables, ideales para modelos de ensamble como Random Forest.

In [37]:
# =====================================================================
# 4. ESPECIALIDAD_MEDICA (Versión Final: Sin Micro-Clases y Anti-Mojibake)
# =====================================================================

def clasificar_especialidad(esp):
    esp = str(esp).upper().strip()

    # ---------------- ONCOLOGÍA ----------------
    if any(k in esp for k in ["ONCOLOG", "RADIOTERAPIA", "HEMATOLOGAA ONCOLOG", "HEMATOLOGIA ONCOLOG"]):
        return "ONCOLOGIA"
        
    # ---------------- APOYO CLÍNICO Y DIAGNÓSTICO ----------------
    elif any(k in esp for k in ["IMAGENOLOG", "RADIOLOG", "NEURORRADIOLOG", "LABORATORIO", "MICROBIO", "ANATOM", "PATOLOG", "NUCLEAR", "TRANSFUSIONAL", "GENETICA"]):
        return "APOYO_CLINICO_DIAGNOSTICO"

    # ---------------- PEDIATRÍA Y MATERNIDAD ----------------
    elif any(k in esp for k in ["PEDIATR", "NEONATOLOG", "ADOLESCENCIA"]):
        return "PEDIATRIA"
    elif any(k in esp for k in ["GINECO", "OBSTETRIC", "MATRONA", "FETAL"]):
        return "GINECO_OBSTETRICIA"

    # ---------------- ÁREA QUIRÚRGICA ----------------
    elif "CIRUG" in esp and "GENERAL" in esp:
        return "CIRUGIA_GENERAL"
    elif any(k in esp for k in ["CIRUG", "UROLOG", "OTORRINO", "OFTALMO", "COLOPROCTOLOG", "ANESTESIOLOG"]):
        return "ESPECIALIDADES_QUIRURGICAS"
    
    elif "TRAUMATOLOG" in esp or "ORTOPEDIA" in esp:
        return "TRAUMATOLOGIA"

    # ---------------- CUIDADOS CRÍTICOS Y URGENCIA ----------------
    elif "URGENCIA" in esp:
        return "URGENCIA"
    elif "INTENSIV" in esp:
        return "CUIDADOS_CRITICOS"

    # ---------------- ODONTOLOGÍA ----------------
    elif any(k in esp for k in ["ODONTO", "DENT", "MAXILO", "ENDODON", "PERIODON", "TEMPOROMAND", "IMPLANTOLOG", "ORAL"]):
        return "ODONTOLOGIA"

    # ---------------- MEDICINA GENERAL ----------------
    # Cazando todas las fugas de encoding para "Médico"
    elif any(k in esp for k in ["MEDICINA GENERAL", "MEDICO GENERAL", "FAMILIAR", "MÃ\x89DICO GENERAL", "MÉDICO GENERAL", "MADICO GENERAL", "MÃ‰DICO GENERAL"]):
        return "MEDICINA_GENERAL"

    # ---------------- ESPECIALIDADES MÉDICAS (Volumen Medio/Alto) ----------------
    elif "CARDIOLOG" in esp or "CARDIOVASCULAR" in esp:
        return "CARDIOVASCULAR"
    elif "RESPIRATOR" in esp or "BRONCOPULMONAR" in esp:
        return "RESPIRATORIO"
    elif "GASTRO" in esp or "DIGESTIV" in esp:
        return "DIGESTIVO"
    elif "NEFROLOG" in esp:
        return "RENAL"
    elif "NEUROLOG" in esp:
        return "NEUROLOGIA"
    elif "PSIQUIATR" in esp:
        return "PSIQUIATRIA"
    elif "HEMATOLOG" in esp:
        return "HEMATOLOGIA"

    # ---------------- NULOS ----------------
    elif esp in ["DESCONOCIDO", "NAN", "NONE", "NULL", ""]:
        return "DESCONOCIDO"

    # ---------------- MEDICINA INTERNA Y MICRO-SUBS ----------------
    # El "Else" ahora es un sumidero seguro para Medicina Interna y sus micro-ramas
    # (Absorbe Geriatría, Dermatología, Reumatología, Endocrinología, Nutriología, Rehabilitación, Inmuno/Infecto)
    else:
        return "MEDICINA_INTERNA_Y_SUBS"

# Construcción de diccionario y simulación (igual que antes)
especialidades_unicas = set()
for archivo in archivos_procesados:
    df_temp = pd.read_csv(os.path.join(carpeta_procesados, archivo), usecols=["ESPECIALIDAD_MEDICA"], low_memory=False)
    especialidades_unicas.update(df_temp["ESPECIALIDAD_MEDICA"].dropna().str.strip().str.upper().unique())

diccionario_especialidad = {esp: clasificar_especialidad(esp) for esp in especialidades_unicas}
nulos_especialidad = ["DESCONOCIDO", "NAN", "NONE", "NULL", ""]

frecuencias, impactos = simular_transformacion_categorica(
    carpeta=carpeta_procesados, 
    archivos=archivos_procesados, 
    columna="ESPECIALIDAD_MEDICA", 
    mapeo=diccionario_especialidad, 
    valores_nulos=nulos_especialidad
)
mostrar_resultados(frecuencias, impactos, "ESPECIALIDAD_MEDICA")


[ESPECIALIDAD_MEDICA] Simulando mapeo y evaluando impacto de nulos...

Resultados para variable: ESPECIALIDAD_MEDICA

Frecuencias por Cohorte:
- Cohorte Total:


,2019,2020,2021,2022,2023,2024
MAPPED,,,,,,
CARDIOVASCULAR,22662,15186,19430,22205,24952,27158
CIRUGIA_GENERAL,181028,122904,119369,137305,149692,162132
DIGESTIVO,3978,1674,1579,2170,2841,2739
GINECO_OBSTETRICIA,233025,181753,174103,204148,210571,206490
HEMATOLOGIA,8569,1234,1338,1762,2098,2890
MEDICINA_INTERNA_Y_SUBS,430048,306491,338727,368960,417978,450164
ODONTOLOGIA,1439,190,341,637,1251,1038
ONCOLOGIA,22394,11842,10370,11448,12817,13697
PEDIATRIA,105241,61383,65978,90750,104456,99959


- Cohorte Oncológica:


,2019,2020,2021,2022,2023,2024
MAPPED,,,,,,
CARDIOVASCULAR,432,326,459,610,738,827
CIRUGIA_GENERAL,21885,16796,17532,19428,21448,22213
DIGESTIVO,198,129,159,194,262,329
GINECO_OBSTETRICIA,6752,5585,6105,7280,8319,9005
HEMATOLOGIA,7457,1063,1197,1520,1816,2459
MEDICINA_INTERNA_Y_SUBS,34159,27002,29783,35065,42070,46966
ODONTOLOGIA,12,3,4,6,5,5
ONCOLOGIA,21806,11228,9888,10800,12258,13052
PEDIATRIA,1164,1033,1166,1187,1145,1218


- Cohorte Control:


...

,Faltantes
Año,
2019,0
2020,0
2021,0
2022,0
2023,0
2024,0


- Cohorte Oncológica:


,Faltantes
Año,
2019,0
2020,0
2021,0
2022,0
2023,0
2024,0


- Cohorte Control:


,Faltantes
Año,
2019,0
2020,0
2021,0
2022,0
2023,0
2024,0


In [33]:
aplicar_transformacion_categorica(carpeta=carpeta_procesados, archivos=archivos_procesados, columna="ESPECIALIDAD_MEDICA", 
                                  mapeo=diccionario_especialidad, valores_nulos=nulos_especialidad)


[ESPECIALIDAD_MEDICA] EJECUTANDO TRANSFORMACIÓN Y LIMPIEZA DEFINITIVA...
-> SOBRESCRIBIENDO 'ESPECIALIDAD_MEDICA'
  -> 2019 | Inicial: 1,126,042 | Eliminadas: 31 | Final: 1,126,011
  -> 2020 | Inicial: 776,762 | Eliminadas: 20 | Final: 776,742
  -> 2021 | Inicial: 813,241 | Eliminadas: 0 | Final: 813,241
  -> 2022 | Inicial: 932,569 | Eliminadas: 0 | Final: 932,569
  -> 2023 | Inicial: 1,038,434 | Eliminadas: 0 | Final: 1,038,434
  -> 2024 | Inicial: 1,084,635 | Eliminadas: 0 | Final: 1,084,635
RESUMEN GLOBAL DE IMPACTO:
  Total filas analizadas: 5,771,683
  Total filas eliminadas: 51
  Total filas conservadas: 5,771,632


,2019,2020,2021,2022,2023,2024
ESPECIALIDAD_MEDICA,,,,,,
APOYO_CLINICO_DIAGNOSTICO,2241,707,842,1300,1546,1715
CARDIOVASCULAR,22662,15186,19430,22205,24952,27158
CIRUGIA_GENERAL,181031,122904,119369,137305,149692,162132
CUIDADOS_CRITICOS,8001,11816,16320,13740,13468,14909
DIGESTIVO,3978,1674,1579,2170,2841,2739
ESPECIALIDADES_QUIRURGICAS,238900,142819,160278,209937,248100,261765
GINECO_OBSTETRICIA,233025,181753,174103,204148,210571,206490
HEMATOLOGIA,8569,1234,1338,1762,2098,2890
MEDICINA_GENERAL,24886,13313,10252,8653,7810,9251


## Evaluación y Procesamiento de Variable Operativa: Tipo de Actividad (TIPO_ACTIVIDAD)

### 1. Descripción y Justificación
La variable `TIPO_ACTIVIDAD` clasifica la modalidad de atención del episodio clínico, distinguiendo la naturaleza del consumo de infraestructura hospitalaria. El análisis de esta variable detectó anomalías de registro críticas (quiebres temporales) que requerían una consolidación financiera:

* **Desaparición de Categorías (Sesgo Temporal):** Las categorías `HOSPITALIZACION EN URGENCIA` y `HOSPITALIZACION DIURNA` desaparecen abruptamente de la base de datos después del año 2019. Esto obedece a un cambio en los criterios de codificación del MINSAL (sistema SIGTE/GRD), no a una desaparición real de las prácticas clínicas. Si no se corrigen, los modelos predictivos asociarían erróneamente ciertas características clínicas exclusivamente al año 2019.
* **Corrupción de Caracteres (Mojibake):** Se integraron todas las posibles variantes de codificación defectuosa (ej. *CIRUGÃA*, *HOSPITALIZACIÃ“N*) para evitar la pérdida de cientos de miles de registros.

### 2. Estrategia de Transformación (Ingeniería de Características)
Se diseñó un mapeo binarizado que refleja las dos grandes dinámicas estructurales de costos (GRD) de un hospital: con o sin pernoctación.

* **HOSPITALIZACION_TRADICIONAL:** Absorbe la categoría general de `HOSPITALIZACION` y la extinta `HOSPITALIZACION EN URGENCIA`. En el sistema GRD, si un paciente cruza la medianoche (00:00 hrs) recibiendo cuidados continuos, consume un "Día Cama", sin importar si físicamente pernocta en una sala de dotación o en una camilla de observación de urgencias. Representa episodios de alto costo hotelero.
* **AMBULATORIO_CMA:** Fusiona la `CIRUGIA MAYOR AMBULATORIA (CMA)` con la extinta `HOSPITALIZACION DIURNA`. Administrativamente, la Hospitalización Diurna implica que el paciente ingresa, recibe un tratamiento complejo (ej. infusión de quimioterapia) y es dado de alta el mismo día calendario. Se agrupa con CMA porque ambas representan **alta complejidad médica sin consumo de pernoctación** (Días Cama = 0).

### 3. Análisis de Resultados de la Simulación

**Distribución de Frecuencias:**
La agrupación expone una tendencia clave en la gestión de pacientes con cáncer. La actividad resolutiva sin pernoctación (**AMBULATORIO_CMA**) muestra una clara expansión post-pandemia, pasando de ~3.600 casos en 2020 a más de 11.000 en 2024. Esta variable binarizada actuará como un predictor estructural crítico para el modelo de Consumo de Recursos, aislando los episodios ambulatorios de las internaciones prolongadas.

**Impacto en Variables Objetivo (Nulos):**
La simulación de eliminación de nulos arrojó resultados óptimos:
* **Cohorte Oncológica:** Cero registros perdidos. La integridad de los datos de cáncer es del **100%**.
* **Grupo Control:** Apenas se eliminaron **3 registros en el año 2019**, una cifra estadísticamente insignificante sobre una base histórica de varios millones de egresos.
* **Veredicto Final:** **Aprobado para Transformación y Eliminación de Nulos**. La variable ahora es temporalmente coherente, algorítmicamente segura y su binarización refleja con precisión el peso financiero de la estadía clínica.

In [34]:
# =====================================================================
# TIPO_ACTIVIDAD (Consolidación Operativa)
# =====================================================================
# Esta variable tiene un problema similar al de procedencia: "HOSPITALIZACION EN URGENCIA" 
# y "HOSPITALIZACION DIURNA" desaparecen después de 2019. Ambas son formas de 
# "HOSPITALIZACION", por lo que las agruparemos para evitar quiebres temporales.

diccionario_actividad = {
    "HOSPITALIZACION": "HOSPITALIZACION_TRADICIONAL",
    "HOSPITALIZACIÃN": "HOSPITALIZACION_TRADICIONAL",
    "HOSPITALIZACIÓN": "HOSPITALIZACION_TRADICIONAL",
    "HOSPITALIZACION EN URGENCIA": "HOSPITALIZACION_TRADICIONAL",
    "HOSPITALIZACIÃN EN URGENCIA": "HOSPITALIZACION_TRADICIONAL",
    "HOSPITALIZACION DIURNA": "AMBULATORIO_CMA",
    "HOSPITALIZACIÃN DIURNA": "AMBULATORIO_CMA",
    "CIRUGIA MAYOR AMBULATORIA (CMA)": "AMBULATORIO_CMA",
    "CIRUGÃA MAYOR AMBULATORIA (CMA)": "AMBULATORIO_CMA",
    "CIRUGÍA MAYOR AMBULATORIA (CMA)": "AMBULATORIO_CMA",
    
    # Nulos
    "DESCONOCIDO": "DESCONOCIDO",
    "NO IDENTIFICADO": "DESCONOCIDO"
}

nulos_actividad = ["DESCONOCIDO", "NAN", "NONE", "NULL", ""]

frecuencias, impactos = simular_transformacion_categorica(
    carpeta=carpeta_procesados,
    archivos=archivos_procesados, 
    columna="TIPO_ACTIVIDAD", 
    mapeo=diccionario_actividad, 
    valores_nulos=nulos_actividad
)
mostrar_resultados(frecuencias, impactos, "TIPO_ACTIVIDAD") 


[TIPO_ACTIVIDAD] Simulando mapeo y evaluando impacto de nulos...

Resultados para variable: TIPO_ACTIVIDAD

Frecuencias por Cohorte:
- Cohorte Total:


,2019,2020,2021,2022,2023,2024
MAPPED,,,,,,
AMBULATORIO_CMA,196655,79381,100805,154742,193525,212007
HOSPITALIZACION_TRADICIONAL,929353,697361,712436,777827,844909,872628
NULOS_A_ELIMINAR,3,0,0,0,0,0


- Cohorte Oncológica:


,2019,2020,2021,2022,2023,2024
MAPPED,,,,,,
HOSPITALIZACION_TRADICIONAL,76066,61025,62197,70140,80447,87965
AMBULATORIO_CMA,20379,3635,5674,7908,10213,11079


- Cohorte Control:


...

,Faltantes,MORT(0),MORT(1),SEV(0),SEV(1),SEV(2),SEV(3),CONS(0),CONS(1),CONS(2)
Año,,,,,,,,,,
2019,3,3,0,3,0,0,0,3,0,0
2020,0,0,0,0,0,0,0,0,0,0
2021,0,0,0,0,0,0,0,0,0,0
2022,0,0,0,0,0,0,0,0,0,0
2023,0,0,0,0,0,0,0,0,0,0
2024,0,0,0,0,0,0,0,0,0,0


- Cohorte Oncológica:


,Faltantes
Año,
2019,0
2020,0
2021,0
2022,0
2023,0
2024,0


- Cohorte Control:


,Faltantes,MORT(0),MORT(1),SEV(0),SEV(1),SEV(2),SEV(3),CONS(0),CONS(1),CONS(2)
Año,,,,,,,,,,
2019,3,3,0,3,0,0,0,3,0,0
2020,0,0,0,0,0,0,0,0,0,0
2021,0,0,0,0,0,0,0,0,0,0
2022,0,0,0,0,0,0,0,0,0,0
2023,0,0,0,0,0,0,0,0,0,0
2024,0,0,0,0,0,0,0,0,0,0


In [35]:
aplicar_transformacion_categorica(carpeta=carpeta_procesados, archivos=archivos_procesados, columna="TIPO_ACTIVIDAD", 
                                  mapeo=diccionario_actividad, valores_nulos=nulos_actividad)


[TIPO_ACTIVIDAD] EJECUTANDO TRANSFORMACIÓN Y LIMPIEZA DEFINITIVA...
-> SOBRESCRIBIENDO 'TIPO_ACTIVIDAD'
  -> 2019 | Inicial: 1,126,011 | Eliminadas: 3 | Final: 1,126,008
  -> 2020 | Inicial: 776,742 | Eliminadas: 0 | Final: 776,742
  -> 2021 | Inicial: 813,241 | Eliminadas: 0 | Final: 813,241
  -> 2022 | Inicial: 932,569 | Eliminadas: 0 | Final: 932,569
  -> 2023 | Inicial: 1,038,434 | Eliminadas: 0 | Final: 1,038,434
  -> 2024 | Inicial: 1,084,635 | Eliminadas: 0 | Final: 1,084,635
RESUMEN GLOBAL DE IMPACTO:
  Total filas analizadas: 5,771,632
  Total filas eliminadas: 3
  Total filas conservadas: 5,771,629


,2019,2020,2021,2022,2023,2024
TIPO_ACTIVIDAD,,,,,,
AMBULATORIO_CMA,196655,79381,100805,154742,193525,212007
HOSPITALIZACION_TRADICIONAL,929353,697361,712436,777827,844909,872628


## Evaluación y Procesamiento de Variable Clínica: Servicio de Ingreso (SERVICIOINGRESO)

### 1. Descripción y Justificación
La variable `SERVICIOINGRESO` detalla el tipo de cama o unidad física donde el paciente fue admitido. El análisis exploratorio reveló tres desafíos fundamentales que inhabilitaban su uso directo en modelos predictivos:

* **Cardinalidad Extrema y Fragmentación:** La base presentaba 72 categorías distintas de camas. Muchas eran divisiones administrativas redundantes para el ajuste de riesgo clínico (ej. separar "NEONATOLOGIA CUNAS" de "NEONATOLOGIA INCUBADORAS" o "HOSPITAL DE DÍA" de "AMBULATORIO").
* **Corrupción de Caracteres (Mojibake Severo):** La exportación histórica del MINSAL (pre-2022) presentaba fallas de codificación graves donde las tildes destruían la estructura de la palabra (ej. *MÉDICO* registrado como *MÃ‰DICO*, o *ONCOLÓGICO* como *ONCOLÃ“GICO*). Un mapeo estricto dejaba cientos de miles de registros en categorías residuales inclasificables.
* **Aumento Atípico de Nulos Post-Pandemia:** Se detectó un incremento de registros clasificados como "DESCONOCIDO" a partir de 2021-2022, un artefacto asociado a la habilitación masiva de camas indiferenciadas por la pandemia de COVID-19.

### 2. Estrategia de Transformación (Ingeniería de Características)
Para resolver la cardinalidad y el *mojibake*, se diseñó un algoritmo de búsqueda por **raíces semánticas (*stemming*)** robusto ante caracteres corruptos. El algoritmo logró colapsar las 72 variaciones en **8 Macro-Áreas Estructurales** basadas en el nivel de consumo de recursos:

* **CUIDADOS_CRITICOS:** Camas de alta complejidad técnica y soporte vital (UCI, UTI, Coronaria, Quemados).
* **AREA_QUIRURGICA:** Camas de pre y post operatorio, pabellones, cirugía mayor ambulatoria (CMA) y especialidades quirúrgicas puras (Otorrinolaringología, Maxilofacial).
* **MEDICINA_INTERNA:** Camas básicas y de cuidados medios (medicina general, derivación médico-quirúrgica, pensionados, geriatría).
* **GINECO_OBSTETRICIA:** Maternidad, puerperio y patología obstétrica.
* **PEDIATRIA:** Todas las unidades infantiles y neonatales.
* **ONCOLOGIA:** Unidades dedicadas exclusivamente a quimioterapia/radioterapia y hospitales de día oncológicos.
* **NEURO_PSIQUIATRIA:** Camas de corta estadía psiquiátrica y agudos neurológicos.
* **DESCONOCIDO:** Consolidación de nulos y categorías residuales sin volumen estadístico ("OTRA").

### 3. Análisis de Resultados de la Simulación

**Distribución de Frecuencias:**
El uso de *stemming* anti-mojibake rescató más de 300.000 registros anuales. La distribución oncológica (2024) refleja perfectamente el circuito clínico del cáncer: el **48.5% (45.799 casos)** ingresa al `AREA_QUIRURGICA` para resolución quirúrgica, el **20.8% (19.686 casos)** a `MEDICINA_INTERNA` para manejo de complicaciones, y un volumen crítico de **13.174 casos** requiere `CUIDADOS_CRITICOS`. 

**Impacto en Variables Objetivo (Nulos):**
La eliminación de la etiqueta `NULOS_A_ELIMINAR` (nulos y desconocidos) fue auditada para descartar sesgos:
* **Frecuencia:** Los nulos ascienden gradualmente post-pandemia, alcanzando **723 episodios oncológicos en 2024** (0.7% del total de la cohorte).
* **Ausencia de Sesgo en Mortalidad:** Al auditar los 723 casos eliminados en 2024, la pérdida de pacientes fallecidos es marginal (**34 decesos sobre un total de ~5.000 anuales**, un impacto <0.6%). 
* **Ausencia de Sesgo en Severidad:** La distribución de la Severidad de los casos nulos eliminados (173 leves, 234 moderados, 316 severos) es proporcional a la curva normal del hospital. No se eliminaron concentraciones atípicas de pacientes graves.
* **Veredicto Final:** **Aprobado para Transformación y Eliminación de Nulos**. El volumen de pérdida absoluta (<1%) es estadísticamente imperceptible. Mantener estas filas requeriría imputar un tipo de cama, lo cual destruiría la correlación real entre la intensidad del cuidado (ej. UCI vs Básica) y los costos asociados.

In [40]:
# =====================================================================
# 2. SERVICIOINGRESO (Versión Definitiva Anti-Mojibake)
# =====================================================================

def clasificar_servicio_ingreso(srv):
    srv = str(srv).upper().strip()
    
    # 1. CUIDADOS CRÍTICOS
    if any(kw in srv for kw in ["INTENSIV", "UCI", "INTERMEDIO", "UTI", "CORONARI", "EMERGENCIA", "QUEMADOS", "AGUDOS"]):
        return "CUIDADOS_CRITICOS"
    
    # 2. ONCOLOGÍA
    # Atrapa: ONCOLOGIA, ONCOLÓGICO, ONCOLÃ“GICO, HOSPITA DE DÍA ONCOLÓGICO
    elif any(kw in srv for kw in ["ONCOLOG", "ONCOLÃ", "ONCOLÓ"]):
        return "ONCOLOGIA"
        
    # 3. PEDIATRÍA Y NEONATOLOGÍA
    elif any(kw in srv for kw in ["PEDIATR", "INFANTIL", "NIÑ", "NIÃ", "NEONATOLOG", "CUNA", "INCUBADORA", "LACTANTE", "INFANCIA"]):
        return "PEDIATRIA"

    # 4. GINECO-OBSTETRICIA
    elif any(kw in srv for kw in ["OBSTETR", "GINECOL", "PUERPERIO", "EMBARAZO", "MATERNIDAD", "AREAOBSTETRICA"]):
        return "GINECO_OBSTETRICIA"
        
    # 5. NEURO-PSIQUIATRÍA
    elif any(kw in srv for kw in ["PSIQUIATR", "NEUROL", "FORENSE"]):
        return "NEURO_PSIQUIATRIA"

    # 6. ÁREA QUIRÚRGICA
    elif any(kw in srv for kw in ["CIRUG", "QUIRURG", "QUIRÃ", "QUIRÚRG", "RECUPERACION", "PABELLON", "CMA", "TRAUMATOLOG", "OFTALMOLOG", "UROLOG", "NEUROCIRUG", "MAXILO", "OTORRINO"]):
        return "AREA_QUIRURGICA"
        
    # 7. MEDICINA INTERNA
    elif any(kw in srv for kw in ["MEDICIN", "MEDICA", "MÃ\x89DICA", "MÉDICA", "MEDICO", "MÃ\x89DICO", "MÉDICO", "PENSIONADO", "GERIATR", "TRANSPLANTE", "INFECCIOSOS", "DIABETES"]):
        return "MEDICINA_INTERNA"
        
    # 8. NULOS
    elif srv in ["DESCONOCIDO", "NULOS", "NAN", "NONE", "NULL", "", "OTRA"]:
        return "DESCONOCIDO"
        
    else:
        return srv

# Construimos el diccionario
servicios_unicos = set()
for archivo in archivos_procesados:
    df_temp = pd.read_csv(os.path.join(carpeta_procesados, archivo), usecols=["SERVICIOINGRESO"], low_memory=False)
    servicios_unicos.update(df_temp["SERVICIOINGRESO"].dropna().str.strip().str.upper().unique())

diccionario_servicio_ingreso = {srv: clasificar_servicio_ingreso(srv) for srv in servicios_unicos}
nulos_servicio_ingreso = ["DESCONOCIDO", "NAN", "NONE", "NULL", "", "OTRA"]

frecuencias, impactos = simular_transformacion_categorica(
    carpeta=carpeta_procesados,
    archivos=archivos_procesados, 
    columna="SERVICIOINGRESO", 
    mapeo=diccionario_servicio_ingreso, 
    valores_nulos=nulos_servicio_ingreso
)
mostrar_resultados(frecuencias, impactos, "SERVICIOINGRESO")


[SERVICIOINGRESO] Simulando mapeo y evaluando impacto de nulos...

Resultados para variable: SERVICIOINGRESO

Frecuencias por Cohorte:
- Cohorte Total:


,2019,2020,2021,2022,2023,2024
MAPPED,,,,,,
AREA_QUIRURGICA,394907,269632,299551,370169,437113,477682
CUIDADOS_CRITICOS,187490,122232,130996,134350,149011,155446
GINECO_OBSTETRICIA,221799,176535,167760,193623,195977,187795
MEDICINA_INTERNA,164261,123630,124831,121724,127339,135065
NEURO_PSIQUIATRIA,11797,9543,11081,12404,13380,13395
NULOS_A_ELIMINAR,2781,187,3653,5495,7401,8131
ONCOLOGIA,25621,7687,6337,6469,7308,7791
PEDIATRIA,117352,67296,69032,88335,100905,99330


- Cohorte Oncológica:


,2019,2020,2021,2022,2023,2024
MAPPED,,,,,,
AREA_QUIRURGICA,33534,27548,31666,36200,41804,45799
CUIDADOS_CRITICOS,10371,6907,7128,9433,11880,13174
GINECO_OBSTETRICIA,7256,6051,7001,7970,8893,9342
MEDICINA_INTERNA,17881,14055,12606,14720,17603,19686
NEURO_PSIQUIATRIA,171,145,216,237,205,267
NULOS_A_ELIMINAR,42,9,86,519,580,723
ONCOLOGIA,24164,7238,6001,6104,6957,7393
PEDIATRIA,3026,2707,3167,2865,2738,2660


- Cohorte Control:


...

,Faltantes,MORT(0),MORT(1),SEV(0),SEV(1),SEV(2),SEV(3),CONS(0),CONS(1),CONS(2)
Año,,,,,,,,,,
2019,2781,2779,2,2383,163,122,113,1822,642,317
2020,187,185,2,96,25,24,42,83,27,77
2021,3653,3277,376,43,1257,698,1655,955,923,1775
2022,5495,5183,312,51,1984,1191,2269,1417,1500,2578
2023,7401,7110,291,97,2927,1552,2825,1953,2339,3109
2024,8131,7731,400,38,3243,1867,2983,1867,2563,3701


- Cohorte Oncológica:


,Faltantes,MORT(0),MORT(1),SEV(0),SEV(1),SEV(2),SEV(3),CONS(0),CONS(1),CONS(2)
Año,,,,,,,,,,
2019,42,42,0,30,2,5,5,29,6,7
2020,9,9,0,3,0,3,3,3,2,4
2021,86,71,15,2,13,9,62,4,8,74
2022,519,484,35,6,140,114,259,17,63,439
2023,580,549,31,2,152,114,312,6,80,494
2024,723,689,34,0,173,234,316,20,178,525


- Cohorte Control:


,Faltantes,MORT(0),MORT(1),SEV(0),SEV(1),SEV(2),SEV(3),CONS(0),CONS(1),CONS(2)
Año,,,,,,,,,,
2019,2739,2737,2,2353,161,117,108,1793,636,310
2020,178,176,2,93,25,21,39,80,25,73
2021,3567,3206,361,41,1244,689,1593,951,915,1701
2022,4976,4699,277,45,1844,1077,2010,1400,1437,2139
2023,6821,6561,260,95,2775,1438,2513,1947,2259,2615
2024,7408,7042,366,38,3070,1633,2667,1847,2385,3176


In [41]:
aplicar_transformacion_categorica(carpeta=carpeta_procesados, archivos=archivos_procesados, columna="SERVICIOINGRESO", 
                                  mapeo=diccionario_servicio_ingreso, valores_nulos=nulos_servicio_ingreso)


[SERVICIOINGRESO] EJECUTANDO TRANSFORMACIÓN Y LIMPIEZA DEFINITIVA...
-> SOBRESCRIBIENDO 'SERVICIOINGRESO'
  -> 2019 | Inicial: 1,126,008 | Eliminadas: 2,781 | Final: 1,123,227
  -> 2020 | Inicial: 776,742 | Eliminadas: 187 | Final: 776,555
  -> 2021 | Inicial: 813,241 | Eliminadas: 3,653 | Final: 809,588
  -> 2022 | Inicial: 932,569 | Eliminadas: 5,495 | Final: 927,074
  -> 2023 | Inicial: 1,038,434 | Eliminadas: 7,401 | Final: 1,031,033
  -> 2024 | Inicial: 1,084,635 | Eliminadas: 8,131 | Final: 1,076,504
RESUMEN GLOBAL DE IMPACTO:
  Total filas analizadas: 5,771,629
  Total filas eliminadas: 27,648
  Total filas conservadas: 5,743,981


,2019,2020,2021,2022,2023,2024
SERVICIOINGRESO,,,,,,
AREA_QUIRURGICA,394907,269632,299551,370169,437113,477682
CUIDADOS_CRITICOS,187490,122232,130996,134350,149011,155446
GINECO_OBSTETRICIA,221799,176535,167760,193623,195977,187795
MEDICINA_INTERNA,164261,123630,124831,121724,127339,135065
NEURO_PSIQUIATRIA,11797,9543,11081,12404,13380,13395
ONCOLOGIA,25621,7687,6337,6469,7308,7791
PEDIATRIA,117352,67296,69032,88335,100905,99330


## Evaluación y Procesamiento de Variable Clínica: Procedimiento Principal (PROCEDIMIENTO1)

### 1. Descripción y Justificación
La variable `PROCEDIMIENTO1` registra el código CIE-9-MC de la intervención quirúrgica o terapéutica principal realizada durante el episodio. El análisis de esta variable expuso un desafío clásico de dimensionalidad en datos de facturación médica:

* **Granularidad Inmanejable:** El manual CIE-9-MC contiene miles de códigos específicos, organizados en 100 capítulos macro. Incluso utilizando los 100 capítulos principales, la matriz de datos resultante sufriría de alta dispersión (sparsity), limitando la capacidad de los modelos predictivos para generalizar el costo y la severidad de las intervenciones.
* **Agrupación Anatómico-Funcional:** Desde una perspectiva de gestión hospitalaria (GRD), el consumo de recursos (días de pabellón, insumos, estadía postoperatoria) guarda una estrecha relación con el sistema anatómico intervenido (ej. una cirugía neurológica tiene un perfil de riesgo y costo distinto a una cirugía cutánea).

### 2. Estrategia de Transformación y Mapeo Estructural CIE-9-MC
Se desarrolló un algoritmo basado en expresiones regulares (`Regex`) para extraer el capítulo base de cada código CIE-9-MC (los primeros dos dígitos) y mapearlo a **17 Sistemas Anatómicos y Funcionales**. 

La asignación de capítulos numéricos a cada macro-categoría no fue arbitraria, sino que respeta estrictamente la ontología clínica del manual CIE-9-MC, aplicando correcciones algorítmicas en áreas donde el manual presenta fragmentación. La validación estructural es la siguiente:

* **SISTEMA_NERVIOSO:** Capítulos `01-05` (Incluye cráneo, cerebro, médula espinal y nervios periféricos).
* **SISTEMA_ENDOCRINO:** Capítulos `06-07` (Tiroides, paratiroides y otras glándulas).
* **OFTALMOLOGIA:** Capítulos `08-16` (Estructuras del ojo) y el capítulo `95` (Diagnóstico y tratamiento oftalmológico y otológico menor).
* **OTORRINOLARINGOLOGIA:** Capítulos `18-22` (Oído y vías nasales) y `28-29` (Amígdalas, adenoides y faringe).
* **ODONTOLOGIA_MAXILOFACIAL:** Capítulos `23-27` (Estructuras dentales y bucales). Se incluyó explícitamente el capítulo `76` (Huesos y articulaciones faciales) para evitar su contaminación con traumatología ortopédica, respetando el flujo de cirugía maxilofacial.
* **SISTEMA_RESPIRATORIO:** Capítulos `30-34` (Laringe, tráquea, pulmón y pleura).
* **SISTEMA_CARDIOVASCULAR:** Capítulos `35-39` (Corazón, válvulas y vasos mayores).
* **SISTEMA_HEMATOPOYETICO:** Capítulos `40-41` (Sistema linfático, médula ósea y bazo).
* **SISTEMA_DIGESTIVO:** Capítulos `42-54` (Tracto gastrointestinal completo, desde esófago hasta recto, incluyendo hígado y vías biliares).
* **SISTEMA_URINARIO:** Capítulos `55-59` (Riñón, uréter, vejiga y uretra).
* **REPRODUCTOR_MASCULINO:** Capítulos `60-64` (Próstata, testículos y genitales).
* **REPRODUCTOR_FEMENINO:** Capítulos `65-71` (Ovarios, útero, vagina y vulva).
* **OBSTETRICIA_Y_PARTO:** Capítulos `72-75` (Asistencia al parto y cesáreas).
* **SISTEMA_MUSCULOESQUELETICO:** Capítulos `77-84` (Huesos, articulaciones y tejidos blandos asociados, excluyendo área facial).
* **PIEL_Y_MAMA:** Capítulos `85-86` (Intervenciones en tejido mamario y subcutáneo).
* **IMAGENOLOGIA_Y_DIAGNOSTICO:** Capítulos `87-92` (Radiología, entrevistas clínicas, exámenes microscópicos y medicina nuclear).
* **TERAPEUTICO_Y_NO_QUIRURGICO:** Consolida los capítulos `93-94` y `96-99` (Fisioterapia, salud mental, dispositivos de soporte vital), además de los capítulos residuales `00` y `17` (Procedimientos diversos). Esta categoría aísla eficazmente intervenciones sistémicas sin incisión primaria (ej. quimioterapias y terapias biológicas).

### 3. Análisis de Resultados de la Simulación

**Distribución de Frecuencias:**
La agrupación refleja de manera impecable la epidemiología del hospital. En la cohorte oncológica (2024), destacan tres grandes bloques:
1. **TERAPEUTICO_Y_NO_QUIRURGICO (~25.000 casos):** Absorbe el enorme volumen de quimioterapias y terapias biológicas.
2. **IMAGENOLOGIA_Y_DIAGNOSTICO (~17.900 casos):** Refleja la alta carga de etapificación y control (TACs, PET-CT, biopsias).
3. **SISTEMA_DIGESTIVO (~16.200 casos) y PIEL_Y_MAMA (~13.000 casos):** Evidencia la prevalencia quirúrgica de los cánceres gástricos/colorrectales y de mama, respectivamente.

**Impacto en Variables Objetivo (Nulos):**
La limpieza de la etiqueta `NULOS_A_ELIMINAR` (códigos faltantes o inválidos) requirió un análisis temporal:
* **Mejora de Calidad en el Tiempo:** En los años 2019 y 2020, hubo un déficit de registro importante (912 y 544 casos oncológicos nulos, respectivamente). Sin embargo, a partir de 2021, la calidad del dato mejora drásticamente, promediando **menos de 15 casos oncológicos nulos anuales** (un porcentaje prácticamente de 0%).
* **Evaluación de Sesgo:** La eliminación de los ~900 casos de 2019 no altera estructuralmente las clases minoritarias de las variables objetivo (solo 85 pacientes fallecidos se omiten en una muestra anual de más de 100.000). 
* **Veredicto Final:** **Aprobado para Transformación y Eliminación de Nulos**. La reducción de la dimensionalidad a 17 categorías es metodológicamente óptima. La eliminación de los nulos pre-2021 es la vía más segura para evitar la imputación de procedimientos sintéticos que destruirían la relación real entre la intervención y el costo/severidad.

In [42]:
import re

# =====================================================================
# PROCEDIMIENTO1 (Agrupación CIE-9 a Sistemas Anatómicos)
# =====================================================================

def clasificar_procedimiento_cie9(proc):
    proc_str = str(proc).strip().upper()
    
    # 1. Filtro de nulos
    if proc_str in ["DESCONOCIDO", "NAN", "NONE", "NULL", "S/D", ""]:
        return "DESCONOCIDO"
    
    # 2. Extracción del capítulo CIE-9 (Primeros 1 o 2 dígitos antes del punto o texto)
    # Busca 1 o 2 dígitos al principio del string
    match = re.match(r'^0*(\d{1,2})', proc_str)
    
    if match:
        macro = int(match.group(1))
    else:
        return "SIN_CODIGO_VALIDO"
    
    # 3. Mapeo a Sistemas Anatómicos/Funcionales según CIE-9-MC
    if 1 <= macro <= 5: return "SISTEMA_NERVIOSO"
    elif 6 <= macro <= 7: return "SISTEMA_ENDOCRINO"
    elif (8 <= macro <= 16) or macro == 95: return "OFTALMOLOGIA"
    elif (18 <= macro <= 22) or (28 <= macro <= 29): return "OTORRINOLARINGOLOGIA"
    elif (23 <= macro <= 27) or macro == 76: return "ODONTOLOGIA_MAXILOFACIAL"
    elif 30 <= macro <= 34: return "SISTEMA_RESPIRATORIO"
    elif 35 <= macro <= 39: return "SISTEMA_CARDIOVASCULAR"
    elif 40 <= macro <= 41: return "SISTEMA_HEMATOPOYETICO"
    elif 42 <= macro <= 54: return "SISTEMA_DIGESTIVO"
    elif 55 <= macro <= 59: return "SISTEMA_URINARIO"
    elif 60 <= macro <= 64: return "REPRODUCTOR_MASCULINO"
    elif 65 <= macro <= 71: return "REPRODUCTOR_FEMENINO"
    elif 72 <= macro <= 75: return "OBSTETRICIA_Y_PARTO"
    elif 77 <= macro <= 84: return "SISTEMA_MUSCULOESQUELETICO"
    elif 85 <= macro <= 86: return "PIEL_Y_MAMA"
    elif 87 <= macro <= 92: return "IMAGENOLOGIA_Y_DIAGNOSTICO"
    elif (93 <= macro <= 94) or (96 <= macro <= 99) or macro == 0 or macro == 17: return "TERAPEUTICO_NO_QUIRURGICO"
    else:
        return "TERAPEUTICO_Y_NO_QUIRURGICO"

# 4. Construcción dinámica del diccionario evaluando el dataset
procedimientos_unicos = set()
for archivo in archivos_procesados:
    df_temp = pd.read_csv(os.path.join(carpeta_procesados, archivo), usecols=["PROCEDIMIENTO1"], low_memory=False)
    procedimientos_unicos.update(df_temp["PROCEDIMIENTO1"].dropna().astype(str).str.strip().str.upper().unique())

diccionario_procedimiento = {proc: clasificar_procedimiento_cie9(proc) for proc in procedimientos_unicos}

nulos_procedimiento = ["DESCONOCIDO", "SIN_CODIGO_VALIDO", "NAN", "NONE", "NULL", ""]

# 5. Ejecutar Simulación
frecuencias_proc, impactos_proc = simular_transformacion_categorica(
    carpeta=carpeta_procesados,
    archivos=archivos_procesados, 
    columna="PROCEDIMIENTO1", 
    mapeo=diccionario_procedimiento, 
    valores_nulos=nulos_procedimiento
)

mostrar_resultados(frecuencias_proc, impactos_proc, "PROCEDIMIENTO1")


[PROCEDIMIENTO1] Simulando mapeo y evaluando impacto de nulos...

Resultados para variable: PROCEDIMIENTO1

Frecuencias por Cohorte:
- Cohorte Total:


,2019,2020,2021,2022,2023,2024
MAPPED,,,,,,
IMAGENOLOGIA_Y_DIAGNOSTICO,227088,169740,171796,186237,196487,213120
NULOS_A_ELIMINAR,10923,6740,147,143,170,106
OBSTETRICIA_Y_PARTO,130921,111884,98384,113407,110460,100402
ODONTOLOGIA_MAXILOFACIAL,11598,4938,5689,7147,8978,9813
OFTALMOLOGIA,62061,31504,37663,54991,68545,70327
OTORRINOLARINGOLOGIA,20109,8255,8514,13735,15626,15977
PIEL_Y_MAMA,39146,26365,28035,34090,38671,42524
REPRODUCTOR_FEMENINO,52524,34352,40228,50568,57519,61294
REPRODUCTOR_MASCULINO,24034,11992,13367,19962,22796,24298


- Cohorte Oncológica:


,2019,2020,2021,2022,2023,2024
MAPPED,,,,,,
IMAGENOLOGIA_Y_DIAGNOSTICO,14839,11415,11590,13922,15677,17918
NULOS_A_ELIMINAR,912,544,22,19,16,13
OBSTETRICIA_Y_PARTO,79,76,91,123,118,130
ODONTOLOGIA_MAXILOFACIAL,504,360,362,405,456,528
OFTALMOLOGIA,837,417,591,859,1081,1159
OTORRINOLARINGOLOGIA,516,296,366,415,443,463
PIEL_Y_MAMA,9713,6993,8378,10691,12595,13095
REPRODUCTOR_FEMENINO,2437,2069,2494,2837,3158,3420
REPRODUCTOR_MASCULINO,2005,1419,1539,2016,2341,2355


- Cohorte Control:


...

,Faltantes,MORT(0),MORT(1),SEV(0),SEV(1),SEV(2),SEV(3),CONS(0),CONS(1),CONS(2)
Año,,,,,,,,,,
2019,10923,10361,562,96,5612,3490,1725,4848,3999,2076
2020,6740,6174,566,20,2972,1783,1965,2674,2081,1985
2021,147,145,2,2,76,39,30,72,49,26
2022,143,141,2,4,98,24,17,81,42,20
2023,170,162,8,3,99,40,28,89,55,26
2024,106,104,2,1,51,37,17,45,40,21


- Cohorte Oncológica:


,Faltantes,MORT(0),MORT(1),SEV(0),SEV(1),SEV(2),SEV(3),CONS(0),CONS(1),CONS(2)
Año,,,,,,,,,,
2019,912,827,85,15,250,397,250,124,384,404
2020,544,475,69,0,98,223,223,36,259,249
2021,22,21,1,1,5,7,9,5,8,9
2022,19,19,0,1,6,9,3,2,13,4
2023,16,14,2,0,6,5,5,2,8,6
2024,13,13,0,0,5,6,2,0,11,2


- Cohorte Control:


,Faltantes,MORT(0),MORT(1),SEV(0),SEV(1),SEV(2),SEV(3),CONS(0),CONS(1),CONS(2)
Año,,,,,,,,,,
2019,10011,9534,477,81,5362,3093,1475,4724,3615,1672
2020,6196,5699,497,20,2874,1560,1742,2638,1822,1736
2021,125,124,1,1,71,32,21,67,41,17
2022,124,122,2,3,92,15,14,79,29,16
2023,154,148,6,3,93,35,23,87,47,20
2024,93,91,2,1,46,31,15,45,29,19


In [43]:
aplicar_transformacion_categorica(carpeta=carpeta_procesados, archivos=archivos_procesados, columna="PROCEDIMIENTO1", 
                                  mapeo=diccionario_procedimiento, valores_nulos=nulos_procedimiento, nueva_columna="TIPO_PROCEDIMIENTO")


[PROCEDIMIENTO1] EJECUTANDO TRANSFORMACIÓN Y LIMPIEZA DEFINITIVA...
-> CREANDO NUEVA COLUMNA 'TIPO_PROCEDIMIENTO'
  -> 2019 | Inicial: 1,123,227 | Eliminadas: 10,923 | Final: 1,112,304
  -> 2020 | Inicial: 776,555 | Eliminadas: 6,740 | Final: 769,815
  -> 2021 | Inicial: 809,588 | Eliminadas: 147 | Final: 809,441
  -> 2022 | Inicial: 927,074 | Eliminadas: 143 | Final: 926,931
  -> 2023 | Inicial: 1,031,033 | Eliminadas: 170 | Final: 1,030,863
  -> 2024 | Inicial: 1,076,504 | Eliminadas: 106 | Final: 1,076,398
RESUMEN GLOBAL DE IMPACTO:
  Total filas analizadas: 5,743,981
  Total filas eliminadas: 18,229
  Total filas conservadas: 5,725,752


,2019,2020,2021,2022,2023,2024
TIPO_PROCEDIMIENTO,,,,,,
IMAGENOLOGIA_Y_DIAGNOSTICO,227088,169740,171796,186237,196487,213120
OBSTETRICIA_Y_PARTO,130921,111884,98384,113407,110460,100402
ODONTOLOGIA_MAXILOFACIAL,11598,4938,5689,7147,8978,9813
OFTALMOLOGIA,62061,31504,37663,54991,68545,70327
OTORRINOLARINGOLOGIA,20109,8255,8514,13735,15626,15977
PIEL_Y_MAMA,39146,26365,28035,34090,38671,42524
REPRODUCTOR_FEMENINO,52524,34352,40228,50568,57519,61294
REPRODUCTOR_MASCULINO,24034,11992,13367,19962,22796,24298
SISTEMA_CARDIOVASCULAR,43674,23506,24911,30776,34970,36616


## USOPABELLON
IMAGENOLOGIA_Y_DIAGNOSTICO, OBSTETRICIA_Y_PARTO, ODONTOLOGIA_MAXILOFACIAL, OFTALMOLOGIA, OTORRINOLARINGOLOGIA, PIEL_Y_MAMA, REPRODUCTOR_FEMENINO, REPRODUCTOR_MASCULINO, SISTEMA_CARDIOVASCULAR, SISTEMA_DIGESTIVO, SISTEMA_ENDOCRINO, SISTEMA_HEMATOPOYETICO, SISTEMA_MUSCULOESQUELETICO, SISTEMA_NERVIOSO, SISTEMA_RESPIRATORIO, SISTEMA_URINARIO, TERAPEUTICO_NO_QUIRURGICO

In [44]:
import pandas as pd
import os

def tabla_frecuencias_varias(carpeta, archivos, columnas_categoricas):
    """
    Genera tablas de frecuencias anuales para varias columnas categóricas.
    Retorna un diccionario {columna: DataFrame}
    """
    resultados = {col: {} for col in columnas_categoricas}
    
    for archivo in archivos:
        ruta = os.path.join(carpeta, archivo)
        df = pd.read_csv(ruta, low_memory=False)
        año = archivo[-8:-4]  # extrae el año del nombre
        
        for col in columnas_categoricas:
            if col in df.columns:
                resultados[col][año] = df[col].astype(str).value_counts()
    
    # Combinar resultados en DataFrames
    tablas = {col: pd.DataFrame(resultados[col]).fillna(0).astype(int)
              for col in columnas_categoricas}
    return tablas

# Uso
carpeta_procesados = "../../Datos/Datos procesados"
archivos_procesados = [f"GRD_PROCESADO_{año}.csv" for año in range(2019, 2025)]
columnas = ["MORTALIDAD", "SEVERIDAD", "CONSUMO_RECURSOS"]

tablas = tabla_frecuencias_varias(carpeta_procesados, archivos_procesados, columnas)

# Ejemplo: mostrar tabla de MORTALIDAD
print(tablas["MORTALIDAD"])


               2019    2020    2021    2022     2023     2024
MORTALIDAD                                                   
0           1084011  740684  777884  899894  1006068  1050164
1             28293   29131   31557   27037    24795    26234


In [45]:
print(tablas["SEVERIDAD"])

             2019    2020    2021    2022    2023    2024
SEVERIDAD                                                
0          194396   79292  100767  154694  193431  211978
1          540333  360585  337802  371247  385015  383287
2          233748  173706  182703  211990  251241  266703
3          143827  156232  188169  189000  201176  214430


In [46]:
print(tablas["CONSUMO_RECURSOS"])

                    2019    2020    2021    2022    2023    2024
CONSUMO_RECURSOS                                                
0                 367663  262613  270024  315023  345408  360079
1                 371914  252718  271182  304385  343039  362469
2                 372727  254484  268235  307523  342416  353850


In [47]:
import pandas as pd
import os

def tabla_frecuencias_cancer(carpeta, archivos, columnas_categoricas, filtro_col="CATEGORIA_CANCER", filtro_val="SIN_CANCER"):
    """
    Genera tablas de frecuencias anuales para columnas categóricas,
    considerando solo los casos donde filtro_col != filtro_val.
    Retorna un diccionario {columna: DataFrame}
    """
    resultados = {col: {} for col in columnas_categoricas}
    
    for archivo in archivos:
        ruta = os.path.join(carpeta, archivo)
        df = pd.read_csv(ruta, low_memory=False)
        año = archivo[-8:-4]  # extrae el año del nombre
        
        # Filtrar solo casos con cáncer
        df_filtrado = df[df[filtro_col] != filtro_val]
        
        for col in columnas_categoricas:
            if col in df_filtrado.columns:
                resultados[col][año] = df_filtrado[col].astype(str).value_counts()
    
    # Combinar resultados en DataFrames
    tablas = {col: pd.DataFrame(resultados[col]).fillna(0).astype(int)
              for col in columnas_categoricas}
    return tablas

# Uso
carpeta_procesados = "../../Datos/Datos procesados"
archivos_procesados = [f"GRD_PROCESADO_{año}.csv" for año in range(2019, 2025)]
columnas = ["MORTALIDAD", "SEVERIDAD", "CONSUMO_RECURSOS"]

tablas_cancer = tabla_frecuencias_cancer(carpeta_procesados, archivos_procesados, columnas)

# Ejemplo: mostrar tabla de MORTALIDAD en casos con cáncer
print(tablas_cancer["MORTALIDAD"])
print(tablas_cancer["SEVERIDAD"])
print(tablas_cancer["CONSUMO_RECURSOS"])

             2019   2020   2021   2022   2023   2024
MORTALIDAD                                          
0           90377  59746  63359  72697  84785  92346
1            5114   4361   4404   4813   5279   5962
            2019   2020   2021   2022   2023   2024
SEVERIDAD                                          
0          20338   3632   5671   7901  10211  11079
1          26113  18888  19094  20337  21597  22969
2          29764  23492  23685  26578  31777  34336
3          19276  18095  19313  22694  26479  29924
                   2019   2020   2021   2022   2023   2024
CONSUMO_RECURSOS                                          
2                 64600  37415  36958  50016  57493  57070
1                 20874  22288  24875  20477  23583  31078
0                 10017   4404   5930   7017   8988  10160
